In [2]:
!nvidia-sim

/bin/bash: line 1: nvidia-sim: command not found


In [5]:
# ═══════════════════════════════════════════════════════════════════════════
#  2D PINN — Single-Layer Slope Hydrology  [GEO 4TH INPUT VERSION]
#  Devices 107 (Middle, x=25m) and 108 (Toe, x=7m)
#
#  ORIGINAL FIXES (IoT-domain):
#  ┌─────────────────────────────────────────────────────────────────────┐
#  │ FIX 1: Causal rainfall BC — no future data leakage                  │
#  │ FIX 2: Normalise by T_MAX_TRAIN not full record                     │
#  │ FIX 3: Skill score fixed — same device, correct baseline            │
#  │ FIX 4: FoS spatial scan — sensor positions only + uncertainty label │
#  │ FIX 5: NaN tracking in safe_sq — training halts if >10% NaN        │
#  │ FIX 6: Rain function built from raw record before downsampling      │
#  │ FIX 7: Middle-holdout validation split                              │
#  └─────────────────────────────────────────────────────────────────────┘
#
#  GEO ADDITIONS:
#  ┌─────────────────────────────────────────────────────────────────────┐
#  │ GEO 1: geo (Hz) as 4th network input — trunk is now 5-wide         │
#  │ GEO 2: K_s^eff = Ks*(1 + beta*geo_norm) — vibration modifies Ks   │
#  │ GEO 3: beta per device — learnable, log-parameterised, clamped≥0   │
#  │ GEO 4: log1p min-max normalisation over T_MAX_TRAIN window only    │
#  │ GEO 5: geo interpolated onto every collocation point               │
#  │ GEO 6: beta prior loss — network must earn geo coupling from data   │
#  └─────────────────────────────────────────────────────────────────────┘
# ═══════════════════════════════════════════════════════════════════════════

# ── GOOGLE DRIVE SETUP ───────────────────────────────────────────────────
from google.colab import drive as _gdrive
#_gdrive.mount("/content/drive", force_remount=False)
import os as _os
DRIVE_CKPT_DIR      = "/content/pinn_checkpoints"
DRIVE_CKPT_INTERVAL = 2000
_os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)
print(f"[Drive] Checkpoint directory : {DRIVE_CKPT_DIR}")
print(f"[Drive] Periodic interval    : every {DRIVE_CKPT_INTERVAL} epochs")
# ─────────────────────────────────────────────────────────────────────────

import os, warnings, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as _gs
import matplotlib.patches as _mp
from collections import defaultdict
from torch.autograd import grad as autograd_grad
from scipy.stats import mannwhitneyu
warnings.filterwarnings("ignore")

SEED = 42

def set_seed(s):
    torch.manual_seed(s); np.random.seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Device] {device}")

# ═══════════════════════════════════════════════════════════════════════════
#  MEASURED SITE PARAMETERS
# ═══════════════════════════════════════════════════════════════════════════

X_MAX        = 52.0
T_MAX_FULL   = 2114.0
T_MAX_SYNC   = 1596.0
T_MAX_TRAIN  = T_MAX_SYNC * 0.70
T_MAX        = T_MAX_FULL

G_ACC        = 9.81
RHO_W        = 1000.0
GAMMA_W      = RHO_W * G_ACC

SITE = {
    107: dict(
        x_pos=25.0, sensor_depth_m=0.22, H_m=0.28, slope_deg=24.3,
        theta_r=0.100, theta_s=0.380, alpha=2.70, n_vg=1.23, Ks=2.89e-7,
        rho_b=1643.0, c_prime=0.0, phi_prime=30.0, label="Sandy Clay",
    ),
    108: dict(
        x_pos=7.0, sensor_depth_m=0.30, H_m=0.42, slope_deg=19.86,
        theta_r=0.063, theta_s=0.390, alpha=5.90, n_vg=1.48, Ks=3.64e-6,
        rho_b=1616.0, c_prime=0.0, phi_prime=30.0, label="Sandy Clay Loam",
    ),
}

X_POS_107   = SITE[107]["x_pos"]
X_POS_108   = SITE[108]["x_pos"]
X_NORM_107  = X_POS_107 / X_MAX
X_NORM_108  = X_POS_108 / X_MAX
Z_MAX       = max(SITE[107]["H_m"], SITE[108]["H_m"])

N_HIDDEN  = 4
N_WIDTH   = 64
DROPOUT   = 0.00

CSV_FILE        = "/content/2d_pinn_data_dev_107_108.csv"
DOWNSAMPLE_STEP = 15
SHARED_T0       = 4.58
GAP_THRESH_H    = 2.0
THETA_LO_107 = SITE[107]['theta_r']
THETA_HI_107 = SITE[107]['theta_s']
THETA_LO_108 = SITE[108]['theta_r']
THETA_HI_108 = SITE[108]['theta_s']
RAINFALL_UNIT   = "mm/hr"
RAIN_GATE_MS    = 1e-7

PTF_BOUNDS = {
    107: dict(
        alpha  =(2.70*0.80, 2.70*1.20),
        n_vg   =(1.23*0.80, 1.23*1.20),
        theta_r=(0.100-0.030, 0.100+0.030),
        theta_s=(0.380-0.112, 0.380+0.112),
    ),
    108: dict(
        alpha  =(5.90*0.70, 5.90*1.30),
        n_vg   =(1.48*0.70, 1.48*1.30),
        theta_r=(0.063-0.026, 0.063+0.026),
        theta_s=(0.390-0.140, 0.390+0.140),
    ),
}

# ── GEO 4: log1p normalisation constants (set at load time) ──────────────
GEO_LOG_MIN      = 0.0
GEO_LOG_MAX      = 1.0
GEO_EPS          = 1e-6
GEO_DROPOUT_THRESH = 0.01

def geo_to_norm(geo_hz_tensor):
    """Raw geo (Hz) tensor → normalised [0,1] via log1p min-max."""
    log_g = torch.log1p(torch.clamp(geo_hz_tensor, min=GEO_EPS))
    norm  = (log_g - GEO_LOG_MIN) / max(GEO_LOG_MAX - GEO_LOG_MIN, 1e-9)
    return torch.clamp(norm, 0.0, 1.0)

def geo_to_norm_np(geo_hz_array):
    """Numpy version — used in data loader and inference helpers."""
    log_g = np.log1p(np.clip(geo_hz_array.astype(np.float64), GEO_EPS, None))
    norm  = (log_g - GEO_LOG_MIN) / max(GEO_LOG_MAX - GEO_LOG_MIN, 1e-9)
    return np.clip(norm, 0.0, 1.0).astype(np.float32)

# ── FIX 7 split boundaries ────────────────────────────────────────────────
VAL_LO = 0.35
VAL_HI = 0.65
TE_LO  = 0.85

def _splits(t_h, tr=0.70, vl=0.20):
    t_clip = np.minimum(t_h, T_MAX_SYNC)
    tn     = t_clip / T_MAX_SYNC
    tr_mask  = (tn <= VAL_LO) | (tn > VAL_HI)
    val_mask = (tn > VAL_LO) & (tn <= VAL_HI)
    te_mask  = (tn > TE_LO)  & (t_h <= T_MAX_SYNC)
    return tr_mask, val_mask, te_mask

# ═══════════════════════════════════════════════════════════════════════════
#  PARAMETER INTERPOLATION
# ═══════════════════════════════════════════════════════════════════════════

def _interp_x(x_norm, val107, val108):
    xn107 = X_NORM_107; xn108 = X_NORM_108
    t = torch.clamp((x_norm - xn108) / (xn107 - xn108 + 1e-9), 0.0, 1.0)
    return (1.0 - t) * val108 + t * val107

def get_H(x_norm):
    return _interp_x(x_norm,
                     torch.tensor(SITE[107]["H_m"], dtype=torch.float32),
                     torch.tensor(SITE[108]["H_m"], dtype=torch.float32))

def get_slope_rad(x_norm):
    b107 = np.radians(SITE[107]["slope_deg"])
    b108 = np.radians(SITE[108]["slope_deg"])
    return _interp_x(x_norm,
                     torch.tensor(b107, dtype=torch.float32),
                     torch.tensor(b108, dtype=torch.float32))

def get_rho_b(x_norm):
    return _interp_x(x_norm,
                     torch.tensor(SITE[107]["rho_b"], dtype=torch.float32),
                     torch.tensor(SITE[108]["rho_b"], dtype=torch.float32))

def get_phi_rad(x_norm):
    p107 = np.radians(SITE[107]["phi_prime"])
    p108 = np.radians(SITE[108]["phi_prime"])
    return _interp_x(x_norm,
                     torch.tensor(p107, dtype=torch.float32),
                     torch.tensor(p108, dtype=torch.float32))

# ═══════════════════════════════════════════════════════════════════════════
#  VAN GENUCHTEN
# ═══════════════════════════════════════════════════════════════════════════

def vg_theta(psi, x_norm, params=None):
    if params is not None:
        alpha   = _interp_x(x_norm, params["alpha"][0],   params["alpha"][1])
        n       = _interp_x(x_norm, params["n_vg"][0],    params["n_vg"][1])
        theta_r = _interp_x(x_norm, params["theta_r"][0], params["theta_r"][1])
        theta_s = _interp_x(x_norm, params["theta_s"][0], params["theta_s"][1])
    else:
        alpha   = _interp_x(x_norm,
                    torch.tensor(SITE[107]["alpha"],   dtype=torch.float32, device=psi.device),
                    torch.tensor(SITE[108]["alpha"],   dtype=torch.float32, device=psi.device))
        n       = _interp_x(x_norm,
                    torch.tensor(SITE[107]["n_vg"],    dtype=torch.float32, device=psi.device),
                    torch.tensor(SITE[108]["n_vg"],    dtype=torch.float32, device=psi.device))
        theta_r = _interp_x(x_norm,
                    torch.tensor(SITE[107]["theta_r"], dtype=torch.float32, device=psi.device),
                    torch.tensor(SITE[108]["theta_r"], dtype=torch.float32, device=psi.device))
        theta_s = _interp_x(x_norm,
                    torch.tensor(SITE[107]["theta_s"], dtype=torch.float32, device=psi.device),
                    torch.tensor(SITE[108]["theta_s"], dtype=torch.float32, device=psi.device))
    m   = 1.0 - 1.0 / n
    arg = torch.clamp(alpha * torch.abs(psi), min=0.0)
    Se  = 1.0 / (1.0 + arg.pow(n)).pow(m)
    Se  = torch.where(psi >= 0.0, torch.ones_like(psi), Se)
    Se  = torch.clamp(Se, 1e-6, 1.0 - 1e-6)
    return theta_r + (theta_s - theta_r) * Se

def vg_Se(psi, x_norm, params=None):
    if params is not None:
        alpha = _interp_x(x_norm, params["alpha"][0], params["alpha"][1])
        n     = _interp_x(x_norm, params["n_vg"][0],  params["n_vg"][1])
    else:
        alpha = _interp_x(x_norm,
                    torch.tensor(SITE[107]["alpha"], dtype=torch.float32, device=psi.device),
                    torch.tensor(SITE[108]["alpha"], dtype=torch.float32, device=psi.device))
        n     = _interp_x(x_norm,
                    torch.tensor(SITE[107]["n_vg"],  dtype=torch.float32, device=psi.device),
                    torch.tensor(SITE[108]["n_vg"],  dtype=torch.float32, device=psi.device))
    m   = 1.0 - 1.0 / n
    arg = torch.clamp(alpha * torch.abs(psi), min=0.0)
    Se  = 1.0 / (1.0 + arg.pow(n)).pow(m)
    Se  = torch.where(psi >= 0.0, torch.ones_like(psi), Se)
    return torch.clamp(Se, 1e-6, 1.0)

def vg_K(psi, x_norm, params=None):
    if params is not None:
        alpha = _interp_x(x_norm, params["alpha"][0], params["alpha"][1])
        n     = _interp_x(x_norm, params["n_vg"][0],  params["n_vg"][1])
        Ks    = _interp_x(x_norm, params["Ks"][0],    params["Ks"][1])
    else:
        alpha = _interp_x(x_norm,
                    torch.tensor(SITE[107]["alpha"], dtype=torch.float32, device=psi.device),
                    torch.tensor(SITE[108]["alpha"], dtype=torch.float32, device=psi.device))
        n     = _interp_x(x_norm,
                    torch.tensor(SITE[107]["n_vg"],  dtype=torch.float32, device=psi.device),
                    torch.tensor(SITE[108]["n_vg"],  dtype=torch.float32, device=psi.device))
        Ks    = _interp_x(x_norm,
                    torch.tensor(SITE[107]["Ks"],    dtype=torch.float32, device=psi.device),
                    torch.tensor(SITE[108]["Ks"],    dtype=torch.float32, device=psi.device))
    m   = 1.0 - 1.0 / n
    arg = torch.clamp(alpha * torch.abs(psi), min=0.0)
    Se  = 1.0 / (1.0 + arg.pow(n)).pow(m)
    Se  = torch.where(psi >= 0.0, torch.ones_like(psi), Se)
    Se  = torch.clamp(Se, 1e-6, 1.0 - 1e-6)
    inner = torch.clamp(1.0 - torch.clamp(Se, 1e-6, 1-1e-6).pow(1.0/m), min=0.0)
    K = Ks * Se.pow(0.5) * (1.0 - inner.pow(m)).pow(2.0)
    return torch.clamp(K, 1e-15, 1e-2)

# ── GEO 2: vibration-modified hydraulic conductivity ─────────────────────
def vg_K_eff(psi, x_norm, geo_norm, model_params, beta_params):
    """
    K_s^eff(x, geo) = K(psi,x) * (1 + beta(x) * geo_norm)

    Physical basis: cyclic shear at seismic/vibratory frequencies (0-40 Hz)
    disrupts soil fabric, temporarily raising apparent bulk hydraulic
    conductivity (Manga & Brodsky 2006; Mohr et al. 2015).
    beta is learned — if geo carries no hydrological signal, beta→0.
    """
    K_base = vg_K(psi, x_norm, model_params)
    b107, b108 = beta_params
    alpha_x = torch.clamp(
        (x_norm - X_NORM_108) / (X_NORM_107 - X_NORM_108 + 1e-9), 0.0, 1.0
    )
    beta_x = (1.0 - alpha_x) * b108 + alpha_x * b107
    K_eff  = K_base * (1.0 + beta_x * geo_norm)
    return torch.clamp(K_eff, 1e-15, 1e-2)

def dtheta_dpsi(psi, x_norm, params=None):
    if params is not None:
        alpha   = _interp_x(x_norm, params["alpha"][0],   params["alpha"][1])
        n       = _interp_x(x_norm, params["n_vg"][0],    params["n_vg"][1])
        theta_r = _interp_x(x_norm, params["theta_r"][0], params["theta_r"][1])
        theta_s = _interp_x(x_norm, params["theta_s"][0], params["theta_s"][1])
    else:
        alpha   = _interp_x(x_norm,
                    torch.tensor(SITE[107]["alpha"],   dtype=torch.float32, device=psi.device),
                    torch.tensor(SITE[108]["alpha"],   dtype=torch.float32, device=psi.device))
        n       = _interp_x(x_norm,
                    torch.tensor(SITE[107]["n_vg"],    dtype=torch.float32, device=psi.device),
                    torch.tensor(SITE[108]["n_vg"],    dtype=torch.float32, device=psi.device))
        theta_r = _interp_x(x_norm,
                    torch.tensor(SITE[107]["theta_r"], dtype=torch.float32, device=psi.device),
                    torch.tensor(SITE[108]["theta_r"], dtype=torch.float32, device=psi.device))
        theta_s = _interp_x(x_norm,
                    torch.tensor(SITE[107]["theta_s"], dtype=torch.float32, device=psi.device),
                    torch.tensor(SITE[108]["theta_s"], dtype=torch.float32, device=psi.device))
    m    = 1.0 - 1.0 / n
    dts  = theta_s - theta_r
    arg  = torch.clamp(alpha * torch.abs(psi), min=0.0)
    C    = dts * m * n * alpha * arg.pow(n - 1.0) / (1.0 + arg.pow(n)).pow(m + 1.0)
    C    = torch.where(psi >= 0.0, torch.zeros_like(psi), C)
    return torch.clamp(C, 0.0, 10.0)

# ═══════════════════════════════════════════════════════════════════════════
#  INFINITE SLOPE FoS  (unchanged — geo influences psi, psi drives FoS)
# ═══════════════════════════════════════════════════════════════════════════

def fos_infinite_slope(psi, x_norm):
    H       = get_H(x_norm).to(psi.device)
    beta    = get_slope_rad(x_norm).to(psi.device)
    rho_b   = get_rho_b(x_norm).to(psi.device)
    phi_p   = get_phi_rad(x_norm).to(psi.device)
    gamma_s = rho_b * G_ACC
    gw      = torch.tensor(GAMMA_W, dtype=torch.float32, device=psi.device)
    Se      = vg_Se(psi, x_norm, None)
    u_w     = gw * psi
    u_w_eff = Se * u_w
    u_w_eff = torch.clamp(u_w_eff, min=-2e5, max=2e5)
    sigma_n = gamma_s * H * torch.cos(beta)**2 - u_w_eff
    sigma_n = torch.clamp(sigma_n, min=0.0)
    tau_d   = gamma_s * H * torch.sin(beta) * torch.cos(beta) + 1e-3
    fos = torch.clamp(sigma_n * torch.tan(phi_p) / tau_d, 0.05, 15.0)
    return fos

# ═══════════════════════════════════════════════════════════════════════════
#  PINN MODEL  (GEO 1: 5-input trunk; GEO 3: beta params)
# ═══════════════════════════════════════════════════════════════════════════

class PINNSlope(nn.Module):
    def __init__(self, hidden=N_HIDDEN, width=N_WIDTH, dropout=DROPOUT):
        super().__init__()
        # GEO 1: trunk now takes 5 inputs (x, z, t, q_rain, geo_norm)
        layers = [nn.Linear(5, width), nn.Tanh()]
        for _ in range(hidden - 1):
            layers += [nn.Linear(width, width), nn.Tanh()]
        self.trunk    = nn.Sequential(*layers)
        self.head_psi = nn.Linear(width, 1)

        # VG soil parameters
        self._log_alpha107 = nn.Parameter(torch.log(torch.tensor(SITE[107]["alpha"], dtype=torch.float32)))
        self._log_alpha108 = nn.Parameter(torch.log(torch.tensor(SITE[108]["alpha"], dtype=torch.float32)))
        self._raw_n107     = nn.Parameter(torch.tensor(SITE[107]["n_vg"],    dtype=torch.float32))
        self._raw_n108     = nn.Parameter(torch.tensor(SITE[108]["n_vg"],    dtype=torch.float32))
        self._theta_r107   = nn.Parameter(torch.tensor(SITE[107]["theta_r"], dtype=torch.float32))
        self._theta_r108   = nn.Parameter(torch.tensor(SITE[108]["theta_r"], dtype=torch.float32))
        self._theta_s107   = nn.Parameter(torch.tensor(SITE[107]["theta_s"], dtype=torch.float32))
        self._theta_s108   = nn.Parameter(torch.tensor(SITE[108]["theta_s"], dtype=torch.float32))

        # GEO 3: vibration-Ks coupling scalars beta per device
        # K_s^eff = Ks * (1 + beta * geo_norm)
        # log-parameterised so beta >= 0 always; init at 0.1 (small perturbation)
        self._log_beta107  = nn.Parameter(torch.tensor(np.log(0.1), dtype=torch.float32))
        self._log_beta108  = nn.Parameter(torch.tensor(np.log(0.1), dtype=torch.float32))

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight, gain=1.0)
                nn.init.zeros_(m.bias)

    @property
    def vg_params(self):
        def _clamp_log(param, lo, hi):
            return torch.exp(torch.clamp(param, np.log(lo), np.log(hi)))
        def _clamp(param, lo, hi):
            return torch.clamp(param, lo, hi)
        alpha107 = _clamp_log(self._log_alpha107, *PTF_BOUNDS[107]["alpha"])
        alpha108 = _clamp_log(self._log_alpha108, *PTF_BOUNDS[108]["alpha"])
        n107     = _clamp(self._raw_n107, *PTF_BOUNDS[107]["n_vg"])
        n108     = _clamp(self._raw_n108, *PTF_BOUNDS[108]["n_vg"])
        tr107    = _clamp(self._theta_r107, *PTF_BOUNDS[107]["theta_r"])
        tr108    = _clamp(self._theta_r108, *PTF_BOUNDS[108]["theta_r"])
        ts107    = _clamp(self._theta_s107, *PTF_BOUNDS[107]["theta_s"])
        ts108    = _clamp(self._theta_s108, *PTF_BOUNDS[108]["theta_s"])
        ts107    = torch.max(ts107, tr107.detach() + 0.05)
        ts108    = torch.max(ts108, tr108.detach() + 0.05)
        return dict(
            alpha  =(alpha107, alpha108),
            n_vg   =(n107,     n108),
            theta_r=(tr107,    tr108),
            theta_s=(ts107,    ts108),
            Ks     =(torch.tensor(SITE[107]["Ks"], dtype=torch.float32, device=self._raw_n107.device),
                     torch.tensor(SITE[108]["Ks"], dtype=torch.float32, device=self._raw_n107.device)),
        )

    @property
    def beta_params(self):
        """
        beta clamped to [1e-4, 5.0].
        beta=0 → geo has no effect on Ks.
        beta=5 → 5× Ks amplification at geo_norm=1 (physically extreme ceiling).
        Network will drive beta→0 if geo signal carries no hydraulic information.
        """
        b107 = torch.exp(torch.clamp(self._log_beta107, np.log(1e-4), np.log(5.0)))
        b108 = torch.exp(torch.clamp(self._log_beta108, np.log(1e-4), np.log(5.0)))
        return b107, b108

    def get_learned_params(self):
        p = self.vg_params
        b107, b108 = self.beta_params
        out = {}
        for k, (v107, v108) in p.items():
            out[f"{k}_107"] = float(v107.detach().cpu())
            out[f"{k}_108"] = float(v108.detach().cpu())
        out["beta_107"] = float(b107.detach().cpu())
        out["beta_108"] = float(b108.detach().cpu())
        return out

    def forward(self, x, z, t, q_rain=None, geo_norm=None):
        """
        x, z, t  : as before
        q_rain   : normalised rainfall  (N,1), default zeros
        geo_norm : normalised geo [0,1] (N,1), default zeros
        """
        if q_rain   is None: q_rain   = torch.zeros_like(x)
        if geo_norm is None: geo_norm = torch.zeros_like(x)

        feat  = self.trunk(torch.cat([x, z, t, q_rain, geo_norm], dim=1))
        raw   = self.head_psi(feat)
        psi   = 8.0 * torch.tanh(raw) - 4.0

        p     = self.vg_params
        theta = vg_theta(psi, x, p)
        fos   = fos_infinite_slope(psi, x)
        return psi, theta, fos


# ═══════════════════════════════════════════════════════════════════════════
#  FIX 5 — NaN-aware safe_sq
# ═══════════════════════════════════════════════════════════════════════════

_nan_counts   = defaultdict(int)
_total_counts = defaultdict(int)

def reset_nan_tracker():
    _nan_counts.clear(); _total_counts.clear()

def get_nan_fractions():
    out = {}
    for tag in _nan_counts:
        total = _total_counts.get(tag, 1)
        out[tag] = _nan_counts[tag] / max(total, 1)
    return out

def safe_sq(r, tag=""):
    n_total = r.numel()
    finite_mask = torch.isfinite(r)
    n_nan = n_total - finite_mask.sum().item()
    _nan_counts[tag]   += n_nan
    _total_counts[tag] += n_total
    if n_nan > 0 and n_nan == n_total:
        return torch.tensor(0.0, device=r.device, requires_grad=False)
    r_clean = torch.where(finite_mask, r, torch.zeros_like(r))
    r_clean = torch.clamp(r_clean, -1e3, 1e3)
    n_finite = finite_mask.sum().float()
    v = (r_clean ** 2).sum() / (n_finite + 1e-9)
    if not torch.isfinite(v):
        return torch.tensor(0.0, device=r.device, requires_grad=False)
    return v


# ═══════════════════════════════════════════════════════════════════════════
#  FIX 1 — CAUSAL RAIN FUNCTION
# ═══════════════════════════════════════════════════════════════════════════

_rain_fn = None

def set_rain_fn(fn):
    global _rain_fn
    _rain_fn = fn

def make_causal_rain_fn(t108_norm, q108, t107_norm, q107, xn108, xn107):
    t108_np = t108_norm.astype(np.float64)
    q108_np = q108.astype(np.float64)
    t107_np = t107_norm.astype(np.float64)
    q107_np = q107.astype(np.float64)

    def _rain_fn_impl(x_norm, t_norm):
        t_np = t_norm.detach().cpu().numpy().flatten().astype(np.float64)
        x_np = x_norm.detach().cpu().numpy().flatten().astype(np.float64)
        q8 = np.interp(t_np, t108_np, q108_np, left=0.0, right=float(q108_np[-1]))
        q7 = np.interp(t_np, t107_np, q107_np, left=0.0, right=float(q107_np[-1]))
        al = np.clip((x_np - xn108) / (xn107 - xn108 + 1e-9), 0.0, 1.0)
        q  = np.clip((1 - al) * q8 + al * q7, 0.0, None)
        return torch.tensor(q, dtype=torch.float32,
                            device=t_norm.device).reshape_as(t_norm)
    return _rain_fn_impl


# ═══════════════════════════════════════════════════════════════════════════
#  PHYSICS LOSSES
# ═══════════════════════════════════════════════════════════════════════════

def _grad(y, x):
    return autograd_grad(y, x, grad_outputs=torch.ones_like(y),
                         create_graph=True, retain_graph=True)[0]

def loss_richards(model, x, z, t, geo_norm=None):
    """Richards PDE with vibration-modified Ks (GEO 2)."""
    if geo_norm is None:
        geo_norm = torch.zeros_like(x)

    psi, _, _ = model(x, z, t, geo_norm=geo_norm)
    p            = model.vg_params
    C            = dtheta_dpsi(psi, x, p)
    # GEO 2: use K_eff instead of vg_K
    K            = vg_K_eff(psi, x, geo_norm, p, model.beta_params)
    dpsi_dt_norm = _grad(psi, t)
    dpsi_dz_norm = _grad(psi, z)
    cos_beta     = torch.cos(get_slope_rad(x).to(psi.device))
    flux_z       = K * (dpsi_dz_norm / Z_MAX + cos_beta)
    dflux_dz     = _grad(flux_z, z) / Z_MAX
    T_train_s    = T_MAX_TRAIN * 3600.0
    residual     = C * dpsi_dt_norm / T_train_s - dflux_dz
    residual_scaled = residual * 1e6
    return safe_sq(residual_scaled, "richards")

def vg_Ks(x_norm, params=None):
    Ks107 = torch.tensor(SITE[107]["Ks"], dtype=torch.float32, device=x_norm.device)
    Ks108 = torch.tensor(SITE[108]["Ks"], dtype=torch.float32, device=x_norm.device)
    alpha  = torch.clamp((x_norm - X_NORM_108) / (X_NORM_107 - X_NORM_108 + 1e-9), 0.0, 1.0)
    return Ks108 * (1 - alpha) + Ks107 * alpha

def loss_bc_top(model, x, z, t, geo_norm=None):
    if geo_norm is None:
        geo_norm = torch.zeros_like(x)

    psi, _, _ = model(x, z, t, geo_norm=geo_norm)
    if _rain_fn is None:
        return torch.tensor(0.0, device=x.device)
    p      = model.vg_params
    # GEO 2: K_eff in BC
    K      = vg_K_eff(psi, x, geo_norm, p, model.beta_params)
    q_rain = _rain_fn(x, t)
    Ks_x   = vg_Ks(x, p)
    gate   = (q_rain > RAIN_GATE_MS).float()
    if gate.sum().item() < 2:
        return torch.tensor(0.0, device=x.device, requires_grad=False)
    neumann_mask   = (q_rain <= Ks_x).float()
    dpsi_dz        = _grad(psi, z) / Z_MAX
    cos_beta_slope = torch.cos(get_slope_rad(x).to(psi.device))
    q_pred         = K * (dpsi_dz + cos_beta_slope)
    Ks_ref         = 0.5 * (SITE[107]["Ks"] + SITE[108]["Ks"])
    L_neumann      = safe_sq(gate * neumann_mask * (q_pred - q_rain) / Ks_ref, "bc_top_neu")
    dirichlet_mask = (q_rain > Ks_x).float()
    L_dirichlet    = safe_sq(gate * dirichlet_mask * psi / 1.0, "bc_top_dir")
    return L_neumann + L_dirichlet

def loss_bc_bot(model, x, z, t):
    psi, _, _ = model(x, z, t)
    dpsi_dz = _grad(psi, z) / Z_MAX
    return safe_sq(dpsi_dz, "bc_bot")

PSI_IC_108 = -7.07
PSI_IC_107 = -10.0

def loss_ic(model, x, z, t):
    psi, _, _ = model(x, z, t)
    psi_ic_107 = torch.tensor(PSI_IC_107, dtype=torch.float32, device=x.device)
    psi_ic_108 = torch.tensor(PSI_IC_108, dtype=torch.float32, device=x.device)
    alpha_x    = torch.clamp((x - X_NORM_108) / (X_NORM_107 - X_NORM_108 + 1e-9), 0.0, 1.0)
    psi_ic     = psi_ic_108 * (1 - alpha_x) + psi_ic_107 * alpha_x
    psi_range  = float(abs(PSI_IC_107 - PSI_IC_108) + 1.0)
    return safe_sq((psi - psi_ic) / psi_range, "ic")

def loss_data(model, x_obs, z_obs, t_obs, theta_obs,
              q_rain_obs=None, geo_norm_obs=None, event_weights=None):
    """Data loss — geo_norm_obs passed to forward so trunk sees correct geo."""
    _, theta, _ = model(x_obs, z_obs, t_obs, q_rain_obs,
                        geo_norm=geo_norm_obs)
    residual = theta - theta_obs
    if event_weights is not None:
        return (event_weights * residual ** 2).mean()
    return safe_sq(residual, "data")

def loss_smoothness(model, x, z, t):
    psi, _, _ = model(x, z, t)
    dpsi_dx   = _grad(psi, x) / X_MAX
    d2psi_dx2 = _grad(dpsi_dx, x) / X_MAX
    return safe_sq(d2psi_dx2 * 0.3, "smooth")

def loss_psi_variance(model, x, z, t):
    psi, _, _ = model(x, z, t)
    psi_std = psi.std()
    penalty = torch.relu(0.5 - psi_std) ** 2
    return penalty

def loss_prior_vg(model):
    p = model.vg_params
    loss = torch.tensor(0.0, device=next(model.parameters()).device)
    for key, (v107_ptf, v108_ptf) in [
        ("alpha",   (SITE[107]["alpha"],   SITE[108]["alpha"])),
        ("n_vg",    (SITE[107]["n_vg"],    SITE[108]["n_vg"])),
        ("theta_s", (SITE[107]["theta_s"], SITE[108]["theta_s"])),
    ]:
        p107 = torch.tensor(v107_ptf, dtype=torch.float32, device=loss.device)
        p108 = torch.tensor(v108_ptf, dtype=torch.float32, device=loss.device)
        loss = loss + 0.05 * ((p[key][0] - p107)/p107)**2
        loss = loss + 0.05 * ((p[key][1] - p108)/p108)**2
    return loss

def loss_prior_beta(model, target_beta=0.0, sigma=1.0):
    """
    GEO 6: Gaussian prior beta ~ N(0, sigma^2).
    Forces network to earn geo coupling from data residuals.
    If geo is spuriously correlated with rain, beta stays near 0.
    """
    b107, b108 = model.beta_params
    t = torch.tensor(target_beta, dtype=torch.float32, device=b107.device)
    return ((b107 - t)**2 + (b108 - t)**2) / (sigma**2)


# ═══════════════════════════════════════════════════════════════════════════
#  DATA LOADER  (GEO 4, GEO 5 additions)
# ═══════════════════════════════════════════════════════════════════════════

class SlopeDataLoader:
    def __init__(self, filepath, downsample=DOWNSAMPLE_STEP):
        self.filepath = filepath
        self.ds = downsample

    def load(self):
        global T_MAX_SYNC, T_MAX_FULL, T_MAX, T_MAX_TRAIN
        global GEO_LOG_MIN, GEO_LOG_MAX

        raw = pd.read_csv(self.filepath, low_memory=False)
        raw.columns = [c.strip().lower() for c in raw.columns]
        # Drop trailing empty rows
        raw = raw.dropna(subset=["timestamp"]).copy()
        ts = pd.to_datetime(raw["timestamp"], dayfirst=False, errors="coerce")
        t0 = ts.min()
        raw["t_h"] = (ts - t0).dt.total_seconds() / 3600.0
        raw = raw[ts.notna()].copy()

        df108_raw = raw[raw["devid"]==108].copy().sort_values("t_h").reset_index(drop=True)
        df107_raw = raw[raw["devid"]==107].copy().sort_values("t_h").reset_index(drop=True)

        df108_raw = df108_raw[df108_raw["t_h"] >= SHARED_T0].copy().reset_index(drop=True)
        df107_raw = df107_raw[df107_raw["t_h"] >= SHARED_T0].copy().reset_index(drop=True)
        df108_raw["t_h"] -= SHARED_T0
        df107_raw["t_h"] -= SHARED_T0
        print(f"[StratB] t=0 reset to t_orig={SHARED_T0}h")

        T_MAX_SYNC  = float(df107_raw["t_h"].max())
        T_MAX_FULL  = float(np.ceil(df108_raw["t_h"].max()/6.0)*6.0)
        T_MAX       = T_MAX_FULL
        T_MAX_TRAIN = T_MAX_SYNC * 0.70
        print(f"[Fix2] T_MAX_TRAIN={T_MAX_TRAIN:.1f}h")

        val_lo_h = T_MAX_SYNC * VAL_LO
        val_hi_h = T_MAX_SYNC * VAL_HI
        te_lo_h  = T_MAX_SYNC * TE_LO
        print(f"[Fix7] Train:[0,{val_lo_h:.0f}h)∪({val_hi_h:.0f}h,{T_MAX_SYNC:.0f}h]"
              f"  Val:[{val_lo_h:.0f},{val_hi_h:.0f}h]  Test:[{te_lo_h:.0f},{T_MAX_SYNC:.0f}h]")

        # ── Soil moisture cleaning ────────────────────────────────────────
        for df, did in [(df108_raw, 108), (df107_raw, 107)]:
            s = pd.to_numeric(df["soil"], errors="coerce").ffill().bfill()
            if s.max() > 1.0:
                s = s / 100.0
            theta_r_lit = SITE[did]["theta_r"]
            theta_s_lit = SITE[did]["theta_s"]
            valid = (s >= theta_r_lit).values
            n_dropped = (~valid).sum()
            if n_dropped > 0:
                print(f"[DataClean] Dev{did}: dropping {n_dropped} rows with soil < {theta_r_lit}")
            df.drop(index=df.index[~valid], inplace=True)
            df.reset_index(drop=True, inplace=True)
            s_clean  = s[valid].reset_index(drop=True)
            obs_min  = float(s_clean.min())
            obs_max  = float(s_clean.max())
            s_rescaled = theta_r_lit + (s_clean - obs_min) / (obs_max - obs_min + 1e-9) \
                         * (theta_s_lit - theta_r_lit)
            df["theta"] = s_rescaled.values
            print(f"[DataClean] Dev{did}: rescaled [{obs_min:.4f},{obs_max:.4f}] "
                  f"-> [{theta_r_lit:.4f},{theta_s_lit:.4f}]")
            r = pd.to_numeric(df["rain"], errors="coerce").fillna(0.0).clip(lower=0)
            df["q_ms"] = r.values / 3.6e6

        # ── GEO 4: extract and clean geo, compute log1p norm stats ────────
        for df_, did in [(df107_raw, 107), (df108_raw, 108)]:
            df_["geo_hz"] = pd.to_numeric(df_["geo"], errors="coerce").ffill().bfill().clip(lower=0)
            n_drop = (df_["geo_hz"] < GEO_DROPOUT_THRESH).sum()
            if n_drop > 0:
                df_.loc[df_["geo_hz"] < GEO_DROPOUT_THRESH, "geo_hz"] = np.nan
                df_["geo_hz"] = df_["geo_hz"].interpolate(method="linear").ffill().bfill()
                print(f"[Geo Dev{did}] {n_drop} dropout samples interpolated")
            print(f"[Geo Dev{did}] geo=[{df_['geo_hz'].min():.3f},{df_['geo_hz'].max():.3f}] Hz")

        # Normalisation over T_MAX_TRAIN window ONLY (deployment-safe)
        train_mask107 = df107_raw["t_h"] <= T_MAX_TRAIN
        train_mask108 = df108_raw["t_h"] <= T_MAX_TRAIN
        log_geo_train = np.log1p(np.concatenate([
            df107_raw.loc[train_mask107, "geo_hz"].values,
            df108_raw.loc[train_mask108, "geo_hz"].values,
        ]).clip(GEO_EPS))
        GEO_LOG_MIN = float(log_geo_train.min())
        GEO_LOG_MAX = float(log_geo_train.max())
        print(f"[Geo] log1p norm: min={GEO_LOG_MIN:.4f}  max={GEO_LOG_MAX:.4f}"
              f"  (training window {T_MAX_TRAIN:.0f}h)")

        # Apply to full record (safe for inference beyond T_MAX_TRAIN)
        df107_raw["geo_norm"] = geo_to_norm_np(df107_raw["geo_hz"].values)
        df108_raw["geo_norm"] = geo_to_norm_np(df108_raw["geo_hz"].values)

        # ── FIX 6: rain function from raw record before downsampling ──────
        t108_raw_norm = (df108_raw["t_h"].values / T_MAX_TRAIN).astype(np.float64)
        q108_raw      = df108_raw["q_ms"].values.astype(np.float64)
        t107_raw_norm = (df107_raw["t_h"].values / T_MAX_TRAIN).astype(np.float64)
        q107_raw      = df107_raw["q_ms"].values.astype(np.float64)

        _rain_fn_causal = make_causal_rain_fn(
            t108_raw_norm, q108_raw, t107_raw_norm, q107_raw,
            float(X_NORM_108), float(X_NORM_107)
        )
        set_rain_fn(_rain_fn_causal)
        print("[Fix1] Causal rainfall BC set.")
        print("[Fix6] Rain function from raw record.")

        # ── Gap detection for valid PDE intervals ─────────────────────────
        def _get_segs(t_arr, gap_thresh):
            segs = []
            s = t_arr[0]
            for i in range(1, len(t_arr)):
                if t_arr[i] - t_arr[i-1] > gap_thresh:
                    segs.append((float(s), float(t_arr[i-1]))); s = t_arr[i]
            segs.append((float(s), float(t_arr[-1])))
            return segs

        segs108 = _get_segs(df108_raw["t_h"].values, GAP_THRESH_H)
        segs107 = _get_segs(df107_raw["t_h"].values, GAP_THRESH_H)
        all_segs = sorted(segs108 + segs107, key=lambda x: x[0])
        merged = [list(all_segs[0])]
        for s, e in all_segs[1:]:
            if s - merged[-1][1] <= GAP_THRESH_H:
                merged[-1][1] = max(merged[-1][1], e)
            else:
                merged.append([s, e])

        valid_intervals_norm = []
        for s, e in merged:
            sn = s / T_MAX_TRAIN
            en = min(e, T_MAX_TRAIN) / T_MAX_TRAIN
            if en > sn + 1e-4:
                valid_intervals_norm.append((float(sn), float(en)))

        lengths = np.array([e - s for s, e in valid_intervals_norm])
        weights = lengths / lengths.sum()
        self.valid_intervals_norm = valid_intervals_norm
        self.interval_weights     = weights

        # ── Downsample ────────────────────────────────────────────────────
        df108 = df108_raw.iloc[::self.ds].reset_index(drop=True)
        df107 = df107_raw.iloc[::self.ds].reset_index(drop=True)

        z107 = SITE[107]["sensor_depth_m"] / Z_MAX
        z108 = SITE[108]["sensor_depth_m"] / Z_MAX
        df107["z_norm"] = z107; df107["x_norm"] = X_NORM_107
        df108["z_norm"] = z108; df108["x_norm"] = X_NORM_108
        df107["t_norm"] = df107["t_h"] / T_MAX_TRAIN
        df108["t_norm"] = df108["t_h"] / T_MAX_TRAIN

        self.df107 = df107; self.df108 = df108
        self.df107_raw = df107_raw; self.df108_raw = df108_raw
        self.z_norm_107 = z107; self.z_norm_108 = z108
        self.t_rain_h   = df108_raw["t_h"].values
        self.q_rain_ms  = df108_raw["q_ms"].values

        q_rain_max = max(float(df108_raw["q_ms"].max()),
                         float(df107_raw["q_ms"].max()), 1e-12)
        self.q_rain_max = q_rain_max

        # ── Build tensors ─────────────────────────────────────────────────
        def _interp_rain(df_):
            q = np.interp(df_["t_h"].values,
                          df108_raw["t_h"].values, df108_raw["q_ms"].values,
                          left=0.0, right=float(df108_raw["q_ms"].values[-1]))
            return torch.tensor(q / q_rain_max, dtype=torch.float32,
                                device=device).unsqueeze(1)

        EVENT_WEIGHT = 5.0
        RISE_THRESH  = 0.005

        def _event_weights(df_):
            theta_vals = df_["theta"].values
            dtheta = np.diff(theta_vals, prepend=theta_vals[0])
            weights = np.where(dtheta > RISE_THRESH, EVENT_WEIGHT, 1.0)
            return torch.tensor(weights, dtype=torch.float32,
                                device=device).unsqueeze(1)

        def _tensors(df):
            def _t(c): return torch.tensor(df[c].values, dtype=torch.float32, device=device).unsqueeze(1)
            return _t("x_norm"), _t("z_norm"), _t("t_norm"), _t("theta")

        # GEO 5: geo_norm tensor per device (each device uses its own signal)
        def _geo_tensor(df_downsampled, df_raw_):
            g = np.interp(
                df_downsampled["t_h"].values,
                df_raw_["t_h"].values,
                df_raw_["geo_norm"].values,
                left=float(df_raw_["geo_norm"].values[0]),
                right=float(df_raw_["geo_norm"].values[-1]),
            )
            return torch.tensor(g, dtype=torch.float32, device=device).unsqueeze(1)

        self.x108, self.z108, self.t108, self.th108 = _tensors(df108)
        self.x107, self.z107, self.t107, self.th107 = _tensors(df107)
        self.q108 = _interp_rain(df108)
        self.q107 = _interp_rain(df107)
        self.ew108 = _event_weights(df108)
        self.ew107 = _event_weights(df107)

        # GEO 5: geo tensors for each downsampled device dataset
        self.geo108 = _geo_tensor(df108, df108_raw)
        self.geo107 = _geo_tensor(df107, df107_raw)

        # Save raw geo arrays for collocation interpolation and inference
        self.t_geo107      = df107_raw["t_h"].values.astype(np.float32)
        self.t_geo108      = df108_raw["t_h"].values.astype(np.float32)
        self.geo_norm107   = df107_raw["geo_norm"].values.astype(np.float32)
        self.geo_norm108   = df108_raw["geo_norm"].values.astype(np.float32)

        val_mask_raw = ((df108_raw["t_h"] >= val_lo_h) & (df108_raw["t_h"] <= val_hi_h))
        rain_in_val = float(df108_raw.loc[val_mask_raw, "q_ms"].sum() * 3.6e6)
        print(f"[Fix7] Rainfall in val window: {rain_in_val:.1f} mm"
              f"  {'✓' if rain_in_val > 1.0 else '⚠ dry val window'}")

        print(f"\n[Data] Dev107: {len(df107)} pts"
              f"  θ=[{df107['theta'].min():.3f},{df107['theta'].max():.3f}]"
              f"  geo=[{df107_raw['geo_norm'].min():.3f},{df107_raw['geo_norm'].max():.3f}]")
        print(f"[Data] Dev108: {len(df108)} pts"
              f"  θ=[{df108['theta'].min():.3f},{df108['theta'].max():.3f}]"
              f"  geo=[{df108_raw['geo_norm'].min():.3f},{df108_raw['geo_norm'].max():.3f}]")
        print(f"[Data] T_MAX_SYNC={T_MAX_SYNC:.0f}h  T_MAX_TRAIN={T_MAX_TRAIN:.0f}h")
        return self

# ═══════════════════════════════════════════════════════════════════════════
#  COLLOCATION SAMPLING  (GEO 5: geo interpolated at every collocation point)
# ═══════════════════════════════════════════════════════════════════════════

PDE_NAN_WARN_THRESHOLD = 0.10

def sample_collocation(n_int, n_bc_top, n_bc_bot, n_ic,
                       valid_intervals=None, interval_weights=None,
                       data_loader=None):
    """
    Returns collocation tensors with geo_norm at each point.
    geo_norm is spatially blended between Dev107 and Dev108 signals
    using the same linear interpolation as K_eff.
    """
    def rn(n): return torch.rand(n, 1, device=device, requires_grad=True)
    def ze(n): return torch.zeros(n, 1, device=device, requires_grad=True)
    def on(n): return torch.ones(n, 1, device=device, requires_grad=True)

    def _sample_t_valid(n):
        if valid_intervals is None or len(valid_intervals) == 0:
            return torch.rand(n, 1, device=device)
        seg_idx = np.random.choice(len(valid_intervals), size=n, p=interval_weights)
        t_vals  = np.zeros(n, dtype=np.float32)
        for i, si in enumerate(seg_idx):
            s, e = valid_intervals[si]
            t_vals[i] = s + np.random.rand() * (e - s)
        return torch.tensor(t_vals, dtype=torch.float32, device=device).unsqueeze(1)

    n_each = n_int // 2
    def _jitter_x(val, n):
        base = torch.full((n, 1), val, device=device)
        return (base + 0.05*(torch.rand(n, 1, device=device)-0.5)).clamp(0., 1.)

    x_i  = torch.cat([_jitter_x(X_NORM_107, n_each),
                      _jitter_x(X_NORM_108, n_int-n_each)], dim=0).requires_grad_(True)
    z_i  = rn(n_int)
    t_i  = _sample_t_valid(n_int).requires_grad_(True)

    x_bt = rn(n_bc_top); z_bt = ze(n_bc_top)
    t_bt = _sample_t_valid(n_bc_top).requires_grad_(True)
    x_bb = rn(n_bc_bot); z_bb = on(n_bc_bot)
    t_bb = _sample_t_valid(n_bc_bot).requires_grad_(True)
    x_ic = rn(n_ic); z_ic = rn(n_ic); t_ic = ze(n_ic)

    def _interp_geo(x_t, t_t):
        """
        Blend Dev107 and Dev108 geo_norm at each (x,t) collocation point.
        t_t is in normalised time → convert to hours for interpolation.
        """
        if data_loader is None:
            return torch.zeros(x_t.shape[0], 1, device=device)
        t_h_np = (t_t.detach().cpu().numpy().flatten() * T_MAX_TRAIN).astype(np.float32)
        x_np   = x_t.detach().cpu().numpy().flatten()

        g107 = np.interp(t_h_np, data_loader.t_geo107, data_loader.geo_norm107,
                         left=data_loader.geo_norm107[0],
                         right=data_loader.geo_norm107[-1])
        g108 = np.interp(t_h_np, data_loader.t_geo108, data_loader.geo_norm108,
                         left=data_loader.geo_norm108[0],
                         right=data_loader.geo_norm108[-1])
        # Same spatial blend as K_eff
        alpha = np.clip((x_np - X_NORM_108) / (X_NORM_107 - X_NORM_108 + 1e-9), 0., 1.)
        g_out = (1.0 - alpha) * g108 + alpha * g107
        return torch.tensor(g_out, dtype=torch.float32, device=device).unsqueeze(1)

    geo_i   = _interp_geo(x_i,  t_i)
    geo_bt  = _interp_geo(x_bt, t_bt)
    geo_bb  = _interp_geo(x_bb, t_bb)
    geo_ic  = torch.zeros(n_ic, 1, device=device)  # t=0: no vibration data

    return (
        (x_i,  z_i,  t_i,  geo_i),
        (x_bt, z_bt, t_bt, geo_bt),
        (x_bb, z_bb, t_bb, geo_bb),
        (x_ic, z_ic, t_ic, geo_ic),
    )


# ═══════════════════════════════════════════════════════════════════════════
#  METRICS
# ═══════════════════════════════════════════════════════════════════════════

def _r2(o, p):
    ss = np.sum((o - o.mean())**2)
    return float(1 - np.sum((p - o)**2) / max(ss, 1e-9))

def _rmse(o, p): return float(np.sqrt(np.mean((p - o)**2)))

def compute_metrics_combined(model, data_loader):
    results = {}
    all_obs  = {s: [] for s in ["tr","val","te"]}
    all_pred = {s: [] for s in ["tr","val","te"]}

    for did, x_t, z_t, t_t, th_t, q_t, geo_t, df_ in [
        (108, data_loader.x108, data_loader.z108,
              data_loader.t108, data_loader.th108,
              data_loader.q108, data_loader.geo108,
              data_loader.df108),
        (107, data_loader.x107, data_loader.z107,
              data_loader.t107, data_loader.th107,
              data_loader.q107, data_loader.geo107,
              data_loader.df107),
    ]:
        t_h = df_["t_h"].values
        tr_m, val_m, te_m = _splits(t_h)
        with torch.no_grad():
            _, theta_pred, _ = model(x_t, z_t, t_t, q_t, geo_norm=geo_t)
        obs  = th_t.cpu().numpy().flatten()
        pred = theta_pred.cpu().numpy().flatten()

        for mask, sname in [(tr_m,"tr"),(val_m,"val"),(te_m,"te")]:
            if mask.sum() > 2:
                results[f"r2_{sname}_{did}"]   = _r2(obs[mask],  pred[mask])
                results[f"rmse_{sname}_{did}"] = _rmse(obs[mask], pred[mask])
                results[f"n_{sname}_{did}"]    = int(mask.sum())
                all_obs[sname].append(obs[mask])
                all_pred[sname].append(pred[mask])

    for sname in ["tr","val","te"]:
        if all_obs[sname]:
            oc = np.concatenate(all_obs[sname])
            pc = np.concatenate(all_pred[sname])
            results[f"r2_{sname}"]   = _r2(oc, pc)
            results[f"rmse_{sname}"] = _rmse(oc, pc)
            results[f"n_{sname}"]    = len(oc)
    return results


# ═══════════════════════════════════════════════════════════════════════════
#  TRAINING — STAGE 1
# ═══════════════════════════════════════════════════════════════════════════

def train_stage1(data_loader,
                 n_seeds=2, total_epochs=12000, lr=2e-4,
                 lam_data=500.0,
                 lam_richards=1.0,
                 lam_bc_top=0.1,
                 lam_bc_bot=0.1,
                 lam_ic=0.0,
                 lam_smooth=0.01,
                 lam_prior=0.05,
                 lam_psi_var=1.0,
                 lam_beta=0.01,          # GEO 6: beta prior weight
                 es_patience=10000, anneal_epochs=4000,
                 n_int=2000, n_bc=400, n_ic=400):

    best_model = None
    best_rmse  = np.inf
    best_hist  = None
    all_results = []
    R2_MIN   = 0.55
    RMSE_MAX = 0.03

    for seed in range(n_seeds):
        set_seed(seed + 100)
        model = PINNSlope().to(device)
        opt   = torch.optim.Adam(model.parameters(), lr=lr)
        sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
                    opt, mode='min', factor=0.5, patience=1500, min_lr=5e-6)

        hist  = defaultdict(list)
        best_val_score = -np.inf
        best_val_rmse  = np.inf
        wait  = 0
        best_state    = None
        nan_count     = 0

        print(f"\n{'─'*65}")
        print(f"[Stage1] Seed={seed}  lr={lr}  lam_data={lam_data}"
              f"  lam_pde={lam_richards}  lam_beta={lam_beta}  epochs={total_epochs}")
        print(f"{'─'*65}")

        for ep in range(1, total_epochs+1):
            model.train()
            reset_nan_tracker()

            (x_i,z_i,t_i,geo_i),(x_bt,z_bt,t_bt,geo_bt),\
            (x_bb,z_bb,t_bb,geo_bb),(x_ic,z_ic,t_ic,geo_ic) = \
                sample_collocation(n_int, n_bc, n_bc, n_ic,
                                   valid_intervals=data_loader.valid_intervals_norm,
                                   interval_weights=data_loader.interval_weights,
                                   data_loader=data_loader)

            opt.zero_grad()
            anneal = min(1.0, ep / anneal_epochs)

            # Data loss — pass geo tensors per device
            L_data = (
                lam_data * loss_data(model,
                    data_loader.x108, data_loader.z108,
                    data_loader.t108, data_loader.th108,
                    q_rain_obs=data_loader.q108,
                    geo_norm_obs=data_loader.geo108,
                    event_weights=data_loader.ew108) +
                lam_data * loss_data(model,
                    data_loader.x107, data_loader.z107,
                    data_loader.t107, data_loader.th107,
                    q_rain_obs=data_loader.q107,
                    geo_norm_obs=data_loader.geo107,
                    event_weights=data_loader.ew107)
            )

            L_pde  = anneal * lam_richards * loss_richards(model, x_i,  z_i,  t_i,  geo_norm=geo_i)
            L_bct  = anneal * lam_bc_top   * loss_bc_top(  model, x_bt, z_bt, t_bt, geo_norm=geo_bt)
            L_bcb  = anneal * lam_bc_bot   * loss_bc_bot(  model, x_bb, z_bb, t_bb)
            L_ic   = (anneal * lam_ic * loss_ic(model, x_ic, z_ic, t_ic)
                      if lam_ic > 0.0 else torch.tensor(0.0, device=device))
            L_sm   = anneal * lam_smooth  * loss_smoothness(model, x_i, z_i, t_i)
            L_pr   =          lam_prior   * loss_prior_vg(model)
            L_pvar = anneal * lam_psi_var * loss_psi_variance(model, x_i, z_i, t_i)
            # GEO 6: beta prior
            L_beta =          lam_beta    * loss_prior_beta(model)

            L_total = L_data + L_pde + L_bct + L_bcb + L_ic + L_sm + L_pr + L_pvar + L_beta

            if not torch.isfinite(L_total):
                nan_count += 1; opt.zero_grad(); continue

            L_total.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
            grads_ok = all(torch.isfinite(p.grad).all()
                           for p in model.parameters() if p.grad is not None)
            if grads_ok:
                opt.step()
            else:
                opt.zero_grad(); nan_count += 1

            hist["total"].append(L_total.item())
            hist["data"].append(L_data.item())
            hist["pde"].append(L_pde.item())
            hist["beta"].append(L_beta.item())

            if ep % 500 == 0 or ep == 1:
                model.eval()
                nan_fracs    = get_nan_fractions()
                pde_nan_frac = nan_fracs.get("richards", 0.0)
                pde_status   = "OK" if pde_nan_frac <= PDE_NAN_WARN_THRESHOLD \
                               else f"⚠ NaN={pde_nan_frac:.1%}"

                m      = compute_metrics_combined(model, data_loader)
                r2_val = m.get("r2_val",  float("nan"))
                rm_val = m.get("rmse_val",float("nan"))
                r2_tr  = m.get("r2_tr",   float("nan"))
                rm_tr  = m.get("rmse_tr", float("nan"))

                hist["r2_val"].append((ep, r2_val))
                hist["r2_tr"].append((ep, r2_tr))

                cur_lr     = opt.param_groups[0]["lr"]
                b107_val, b108_val = model.beta_params
                b107_f = float(b107_val.detach().cpu())
                b108_f = float(b108_val.detach().cpu())

                r2_tr_108  = m.get("r2_tr_108",   float("nan"))
                r2_tr_107  = m.get("r2_tr_107",   float("nan"))
                rm_tr_108  = m.get("rmse_tr_108",  float("nan"))
                rm_tr_107  = m.get("rmse_tr_107",  float("nan"))
                r2_val_108 = m.get("r2_val_108",  float("nan"))
                r2_val_107 = m.get("r2_val_107",  float("nan"))
                rm_val_108 = m.get("rmse_val_108", float("nan"))
                rm_val_107 = m.get("rmse_val_107", float("nan"))

                meets_threshold = all(np.isfinite(v) for v in [
                    r2_tr_108, r2_tr_107, rm_tr_108, rm_tr_107,
                    r2_val_108, r2_val_107, rm_val_108, rm_val_107,
                ]) and (
                    r2_tr_108  >= R2_MIN and rm_tr_108  <= RMSE_MAX and
                    r2_tr_107  >= R2_MIN and rm_tr_107  <= RMSE_MAX and
                    r2_val_108 >= R2_MIN and rm_val_108 <= RMSE_MAX and
                    r2_val_107 >= R2_MIN and rm_val_107 <= RMSE_MAX
                )
                val_score = (np.mean([r2_tr_108,r2_tr_107,r2_val_108,r2_val_107])
                             - 10.0*np.mean([rm_tr_108,rm_tr_107,rm_val_108,rm_val_107])
                             ) if meets_threshold else -np.inf

                es_tag = ""
                if np.isfinite(rm_val) and rm_val < best_val_rmse:
                    best_val_rmse = rm_val
                    best_state    = {k: v.clone() for k, v in model.state_dict().items()}
                    es_tag = "  ✓ ckpt"

                if ep % DRIVE_CKPT_INTERVAL == 0:
                    _ckpt_path = os.path.join(DRIVE_CKPT_DIR,
                                              f"stage1_seed{seed}_ep{ep:05d}.pth")
                    torch.save({
                        "epoch": ep, "seed": seed,
                        "state_dict": {k: v.clone().cpu() for k, v in model.state_dict().items()},
                        "r2_val": r2_val, "rmse_val": rm_val, "T_MAX_TRAIN": T_MAX_TRAIN,
                        "GEO_LOG_MIN": GEO_LOG_MIN, "GEO_LOG_MAX": GEO_LOG_MAX,
                    }, _ckpt_path)
                    print(f"  [Drive] {_ckpt_path}")

                rmse_improved  = np.isfinite(rm_val) and rm_val < best_val_rmse + 1e-6
                score_improved = val_score > best_val_score
                if score_improved: best_val_score = val_score
                if rmse_improved or score_improved:
                    wait = 0
                    es_tag += "  ✓ best" if score_improved else ""
                else:
                    wait += 1
                    es_tag += f"  ES {wait*500}/{es_patience}"

                if np.isfinite(rm_val): sched.step(rm_val)

                print(
                    f"  ep={ep:5d}  L={L_total.item():8.4f}"
                    f"  [data={L_data.item():.3f} pde={L_pde.item():.4f}"
                    f"  beta_loss={L_beta.item():.4f}  pde_nan={pde_nan_frac:.1%}]"
                    f"  [{pde_status}]"
                    f"  R²_tr={r2_tr:+.4f}  R²_val={r2_val:+.4f}"
                    f"  RMSE_val={rm_val:.5f}  lr={cur_lr:.1e}"
                    f"  β107={b107_f:.4f}  β108={b108_f:.4f}"
                    f"{es_tag}"
                )
                print(
                    f"         Dev108: R²_tr={r2_tr_108:+.4f}  R²_val={r2_val_108:+.4f}"
                    f"  |  Dev107: R²_tr={r2_tr_107:+.4f}  R²_val={r2_val_107:+.4f}"
                )

                if wait * 500 >= es_patience:
                    print(f"\n  [EarlyStop] Seed={seed}  ep={ep}"
                          f"  best_val_RMSE={best_val_rmse:.5f}")
                    break
                model.train()

        if best_state: model.load_state_dict(best_state)
        model.eval()
        m = compute_metrics_combined(model, data_loader)

        def _dev_passes(did):
            for sname in ["tr","val","te"]:
                r2   = m.get(f"r2_{sname}_{did}",   float("nan"))
                rmse = m.get(f"rmse_{sname}_{did}", float("nan"))
                if not (np.isfinite(r2) and r2 >= R2_MIN and
                        np.isfinite(rmse) and rmse <= RMSE_MAX):
                    return False
            return True

        passes  = _dev_passes(108) and _dev_passes(107)
        r2_tr   = m.get("r2_tr",   float("nan"))
        rm_tr   = m.get("rmse_tr", float("nan"))
        r2_val  = m.get("r2_val",  float("nan"))
        rm_val  = m.get("rmse_val",float("nan"))
        r2_te   = m.get("r2_te",   float("nan"))
        rm_te   = m.get("rmse_te", float("nan"))
        b107_f, b108_f = [float(v.detach().cpu()) for v in model.beta_params]

        status = "✓ PASS" if passes else "✗ FAIL"
        print(f"\n  [Seed {seed} Final] {status}"
              f"  Train R²={r2_tr:.4f} RMSE={rm_tr:.5f}"
              f"  Val R²={r2_val:.4f} RMSE={rm_val:.5f}"
              f"  Test R²={r2_te:.4f} RMSE={rm_te:.5f}"
              f"  β107={b107_f:.4f}  β108={b108_f:.4f}"
              f"  nan_skips={nan_count}")

        saved_state = {k: v.clone().cpu() for k, v in model.state_dict().items()}
        all_results.append(dict(
            seed=seed, state_dict=saved_state, hist=hist,
            r2_tr=r2_tr,   rmse_tr=rm_tr,
            r2_val=r2_val, rmse_val=rm_val,
            r2_te=r2_te,   rmse_te=rm_te,
            r2_tr_108=m.get("r2_tr_108",float("nan")),
            r2_tr_107=m.get("r2_tr_107",float("nan")),
            r2_val_108=m.get("r2_val_108",float("nan")),
            r2_val_107=m.get("r2_val_107",float("nan")),
            rmse_tr_108=m.get("rmse_tr_108",float("nan")),
            rmse_tr_107=m.get("rmse_tr_107",float("nan")),
            rmse_val_108=m.get("rmse_val_108",float("nan")),
            rmse_val_107=m.get("rmse_val_107",float("nan")),
            passes=passes,
            best_state_saved=(best_state is not None),
        ))

    def _absolute_score(r):
        r2_vals  = [r.get(f"r2_{s}_{d}",  float("nan")) for s in ["tr","val"] for d in [108,107]]
        rm_vals  = [r.get(f"rmse_{s}_{d}", float("nan")) for s in ["tr","val"] for d in [108,107]]
        if not all(np.isfinite(v) for v in r2_vals + rm_vals): return -np.inf
        r2_mean   = float(np.mean(r2_vals))
        rmse_mean = float(np.mean(rm_vals))
        r2_score   = np.clip((r2_mean - R2_MIN) / 0.40, 0.0, 1.0)
        rmse_score = np.clip(1.0 - rmse_mean / RMSE_MAX, 0.0, 1.0)
        return 0.5 * r2_score + 0.5 * rmse_score

    def _passes_trainval(r):
        for did in [108,107]:
            for sname in ["tr","val"]:
                r2   = r.get(f"r2_{sname}_{did}",   float("nan"))
                rmse = r.get(f"rmse_{sname}_{did}", float("nan"))
                if not (np.isfinite(r2) and r2 >= R2_MIN and
                        np.isfinite(rmse) and rmse <= RMSE_MAX):
                    return False
        return True

    def _positive_trainval_both(r):
        for did in [108,107]:
            for sname in ["tr","val"]:
                if r.get(f"r2_{sname}_{did}", float("nan")) <= 0.0: return False
        return True

    finite = [r for r in all_results
              if np.isfinite(r["r2_val"]) and np.isfinite(r["rmse_val"])
              and r.get("best_state_saved", True)]
    if not finite:
        raise RuntimeError("All seeds produced NaN metrics.")

    tier1 = [r for r in finite if _passes_trainval(r)]
    tier2 = [r for r in finite if _positive_trainval_both(r)]
    tier3 = [r for r in finite if r.get("r2_val", float("nan")) > 0.0]
    tier4 = finite

    if tier1:   pool, tier_name = tier1, "Tier 1 (threshold pass)"
    elif tier2: pool, tier_name = tier2, "Tier 2 (positive R² both devices)"
    elif tier3: pool, tier_name = tier3, "Tier 3 (positive combined val R²)"
    else:       pool, tier_name = tier4, "Tier 4 (fallback)"

    best_result = max(pool, key=_absolute_score)
    best_model  = PINNSlope().to(device)
    best_model.load_state_dict(
        {k: v.to(device) for k, v in best_result["state_dict"].items()})
    best_model.eval()
    best_hist = best_result["hist"]

    b107_f, b108_f = [float(v.detach().cpu()) for v in best_model.beta_params]
    print(f"\n[Stage1 Best] {tier_name}")
    print(f"  Seed={best_result['seed']}"
          f"  Train R²={best_result['r2_tr']:.4f}  RMSE={best_result['rmse_tr']:.5f}")
    print(f"  Val   R²={best_result['r2_val']:.4f}  RMSE={best_result['rmse_val']:.5f}")
    print(f"  Test  R²={best_result['r2_te']:.4f}  RMSE={best_result['rmse_te']:.5f}")
    print(f"  Learned β: Dev107={b107_f:.4f}  Dev108={b108_f:.4f}")
    if b107_f < 0.05 and b108_f < 0.05:
        print("  [Geo] β≈0 — geo signal did not improve fit beyond rainfall")
    else:
        print(f"  [Geo] Ks_eff(geo_norm=1): "
              f"Dev107={SITE[107]['Ks']*(1+b107_f):.3e}  "
              f"Dev108={SITE[108]['Ks']*(1+b108_f):.3e} m/s")
    return best_model, best_hist, all_results


# ═══════════════════════════════════════════════════════════════════════════
#  STAGE 2 — ROLLING WINDOW ADAPTATION
# ═══════════════════════════════════════════════════════════════════════════

def _persistence_24h_rmse(obs_np, t_h_np):
    if len(obs_np) < 2: return float(np.std(obs_np)) + 1e-6
    dt_h = np.median(np.diff(t_h_np))
    if dt_h <= 0: dt_h = 1.0
    lag_idx = max(1, int(round(24.0 / dt_h)))
    if lag_idx >= len(obs_np): lag_idx = len(obs_np) - 1
    return float(np.sqrt(np.mean((obs_np[lag_idx:] - obs_np[:-lag_idx])**2)))

def train_stage2(model, data_loader,
                 window_h=120, warmup_epochs=500, finetune_epochs=3000,
                 lr=2e-4, lam_data=1000.0, lam_richards=0.5,
                 lam_bc_top=0.1, lam_bc_bot=0.1,
                 lam_smooth=0.01, lam_prior=0.5, lam_beta=0.01,
                 fos_warn_thresh=1.5, es_patience=500):

    model.eval()
    t_sync   = T_MAX_SYNC
    t_starts = np.arange(0, t_sync, window_h / 2)
    window_results = []
    stage1_state   = {k: v.clone() for k, v in model.state_dict().items()}

    print(f"\n{'═'*60}\n[Stage2] {len(t_starts)} windows  window_h={window_h:.0f}h\n{'═'*60}")

    for wi, t_start in enumerate(t_starts):
        t_end = min(t_start + window_h, t_sync)
        if t_end - t_start < window_h * 0.3: continue

        def _slice(df_, x_t, z_t, t_t, th_t):
            t_h_ = df_["t_h"].values
            m = (t_h_ >= t_start) & (t_h_ < t_end)
            if m.sum() < 5: return None,None,None,None,m
            return x_t[m], z_t[m], t_t[m], th_t[m], m

        x108w,z108w,t108w,th108w,m108_win = _slice(
            data_loader.df108, data_loader.x108, data_loader.z108,
            data_loader.t108, data_loader.th108)
        x107w,z107w,t107w,th107w,m107_win = _slice(
            data_loader.df107, data_loader.x107, data_loader.z107,
            data_loader.t107, data_loader.th107)

        if x108w is None: continue
        n107 = 0 if x107w is None else len(x107w)

        def _split_window(x_t, z_t, t_t, th_t, val_frac=0.10):
            if x_t is None: return (None,)*8
            n = len(x_t); n_val = max(1, int(n * val_frac))
            return (x_t[:-n_val], z_t[:-n_val], t_t[:-n_val], th_t[:-n_val],
                    x_t[-n_val:], z_t[-n_val:], t_t[-n_val:], th_t[-n_val:])

        x108tr,z108tr,t108tr,th108tr,x108vl,z108vl,t108vl,th108vl = \
            _split_window(x108w,z108w,t108w,th108w)
        x107tr,z107tr,t107tr,th107tr,x107vl,z107vl,t107vl,th107vl = \
            _split_window(x107w,z107w,t107w,th107w)

        # Geo tensors for this window
        n_val108 = max(1, int(m108_win.sum() * 0.10))
        n_val107 = max(1, int(m107_win.sum() * 0.10)) if x107tr is not None else 0
        geo108_tr = data_loader.geo108[m108_win][:-n_val108]
        geo108_vl = data_loader.geo108[m108_win][-n_val108:]
        geo108_win_full = data_loader.geo108[m108_win]
        geo107_tr = data_loader.geo107[m107_win][:-n_val107] if x107tr is not None else None
        geo107_win_full = data_loader.geo107[m107_win] if x107w is not None else None

        q108_tr   = data_loader.q108[m108_win][:-n_val108]
        ew108_tr  = data_loader.ew108[m108_win][:-n_val108]
        q108_vl_win = data_loader.q108[m108_win][-n_val108:]
        q108_win_full = data_loader.q108[m108_win]
        q107_tr   = data_loader.q107[m107_win][:-n_val107] if x107tr is not None else None
        ew107_tr  = data_loader.ew107[m107_win][:-n_val107] if x107tr is not None else None
        q107_win_full = data_loader.q107[m107_win] if x107w is not None else None

        model.load_state_dict(stage1_state)
        model.train()
        opt = torch.optim.Adam(model.parameters(), lr=lr)
        total_eps = warmup_epochs + finetune_epochs
        best_val = np.inf; wait = 0; best_st = None

        print(f"\n  ── Win{wi+1:02d} [{t_start:.0f}–{t_end:.0f}h]"
              f"  n108={len(x108w)}  n107={n107}  epochs={total_eps} ──")

        for ep in range(total_eps):
            opt.zero_grad()
            L = lam_data * loss_data(model, x108tr, z108tr, t108tr, th108tr,
                                     q_rain_obs=q108_tr,
                                     geo_norm_obs=geo108_tr,
                                     event_weights=ew108_tr)
            if x107tr is not None:
                L = L + lam_data * loss_data(model, x107tr, z107tr, t107tr, th107tr,
                                             q_rain_obs=q107_tr,
                                             geo_norm_obs=geo107_tr,
                                             event_weights=ew107_tr)

            n_each = 400
            def _jx(val, n):
                b = torch.full((n,1), val, device=device)
                return (b + 0.05*(torch.rand(n,1,device=device)-0.5)).clamp(0.,1.)

            x_i = torch.cat([_jx(X_NORM_107,n_each), _jx(X_NORM_108,n_each)],0).requires_grad_(True)
            z_i = torch.rand(n_each*2,1,device=device,requires_grad=True)
            t_i_w = (torch.rand(n_each*2,1,device=device,requires_grad=True)
                     * (t_end - t_start) / T_MAX_TRAIN + t_start / T_MAX_TRAIN)

            # Geo for window collocation
            t_h_w = (t_i_w.detach().cpu().numpy().flatten() * T_MAX_TRAIN).astype(np.float32)
            x_w   = x_i.detach().cpu().numpy().flatten()
            g107w  = np.interp(t_h_w, data_loader.t_geo107, data_loader.geo_norm107,
                               left=data_loader.geo_norm107[0], right=data_loader.geo_norm107[-1])
            g108w  = np.interp(t_h_w, data_loader.t_geo108, data_loader.geo_norm108,
                               left=data_loader.geo_norm108[0], right=data_loader.geo_norm108[-1])
            alp_w  = np.clip((x_w - X_NORM_108)/(X_NORM_107 - X_NORM_108 + 1e-9), 0., 1.)
            geo_i_w = torch.tensor((1-alp_w)*g108w + alp_w*g107w,
                                   dtype=torch.float32, device=device).unsqueeze(1)

            x_bt = torch.cat([_jx(X_NORM_107,100), _jx(X_NORM_108,100)],0).requires_grad_(True)
            z_bt = torch.zeros(200,1,device=device,requires_grad=True)
            t_bt = (torch.rand(200,1,device=device,requires_grad=True)
                    * (t_end - t_start) / T_MAX_TRAIN + t_start / T_MAX_TRAIN)
            t_bt_h = (t_bt.detach().cpu().numpy().flatten() * T_MAX_TRAIN).astype(np.float32)
            x_bt_np = x_bt.detach().cpu().numpy().flatten()
            g107bt = np.interp(t_bt_h, data_loader.t_geo107, data_loader.geo_norm107,
                               left=data_loader.geo_norm107[0], right=data_loader.geo_norm107[-1])
            g108bt = np.interp(t_bt_h, data_loader.t_geo108, data_loader.geo_norm108,
                               left=data_loader.geo_norm108[0], right=data_loader.geo_norm108[-1])
            alp_bt = np.clip((x_bt_np - X_NORM_108)/(X_NORM_107 - X_NORM_108 + 1e-9), 0., 1.)
            geo_bt_w = torch.tensor((1-alp_bt)*g108bt + alp_bt*g107bt,
                                    dtype=torch.float32, device=device).unsqueeze(1)

            x_bb = torch.cat([_jx(X_NORM_107,100), _jx(X_NORM_108,100)],0).requires_grad_(True)
            z_bb = torch.ones(200,1,device=device,requires_grad=True)
            t_bb = (torch.rand(200,1,device=device,requires_grad=True)
                    * (t_end - t_start) / T_MAX_TRAIN + t_start / T_MAX_TRAIN)

            L = L + lam_richards * loss_richards(model, x_i, z_i, t_i_w, geo_norm=geo_i_w)
            L = L + lam_bc_top   * loss_bc_top(  model, x_bt, z_bt, t_bt, geo_norm=geo_bt_w)
            L = L + lam_bc_bot   * loss_bc_bot(  model, x_bb, z_bb, t_bb)
            L = L + lam_smooth   * loss_smoothness(model, x_i, z_i, t_i_w)
            L = L + lam_prior    * loss_prior_vg(model)
            L = L + lam_beta     * loss_prior_beta(model)
            L.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            if ep > warmup_epochs and ep % 200 == 0:
                model.eval()
                with torch.no_grad():
                    _, th_pred_vl, _ = model(x108vl, z108vl, t108vl,
                                             q108_vl_win, geo_norm=geo108_vl)
                rm = _rmse(th108vl.cpu().numpy().flatten(),
                           th_pred_vl.cpu().numpy().flatten())
                if rm < best_val:
                    best_val = rm
                    best_st  = {k: v.clone() for k, v in model.state_dict().items()}
                    wait = 0
                else:
                    wait += 1
                if wait * 200 >= es_patience: break
                model.train()

        if best_st: model.load_state_dict(best_st)
        model.eval()
        t_mid_h = 0.5*(t_start + t_end)

        with torch.no_grad():
            x_107_t = torch.full((1,1), X_NORM_107, device=device)
            x_108_t = torch.full((1,1), X_NORM_108, device=device)
            z_bot   = torch.ones(1,1, device=device)
            t_mid_t = torch.full((1,1), t_mid_h / T_MAX_TRAIN, device=device)

            g107_mid = float(np.interp(t_mid_h, data_loader.t_geo107, data_loader.geo_norm107))
            g108_mid = float(np.interp(t_mid_h, data_loader.t_geo108, data_loader.geo_norm108))
            geo107_t = torch.tensor([[g107_mid]], dtype=torch.float32, device=device)
            geo108_t = torch.tensor([[g108_mid]], dtype=torch.float32, device=device)

            _, _, fos_107 = model(x_107_t, z_bot, t_mid_t, geo_norm=geo107_t)
            _, _, fos_108 = model(x_108_t, z_bot, t_mid_t, geo_norm=geo108_t)
            fos_107_val   = float(fos_107.cpu())
            fos_108_val   = float(fos_108.cpu())
            fos_min_sensed = min(fos_107_val, fos_108_val)

            x_scan  = torch.linspace(X_NORM_108, X_NORM_107, 20, device=device).unsqueeze(1)
            z_b20   = torch.ones(20,1, device=device)
            t_s20   = torch.full((20,1), t_mid_h/T_MAX_TRAIN, device=device)
            x_scan_np = x_scan.cpu().numpy().flatten()
            alp_scan  = np.clip((x_scan_np - X_NORM_108)/(X_NORM_107 - X_NORM_108 + 1e-9), 0., 1.)
            geo_scan  = torch.tensor(
                (1-alp_scan)*g108_mid + alp_scan*g107_mid,
                dtype=torch.float32, device=device).unsqueeze(1)
            _, _, fos_scan = model(x_scan, z_b20, t_s20, geo_norm=geo_scan)
            fos_min_interp = float(fos_scan.min().cpu())

            _, th_pred108, _ = model(x108w, z108w, t108w, q108_win_full,
                                     geo_norm=geo108_win_full)
            rm108 = _rmse(th108w.cpu().numpy().flatten(),
                          th_pred108.cpu().numpy().flatten())
            rm107 = float("nan")
            if x107w is not None:
                _, th_pred107, _ = model(x107w, z107w, t107w, q107_win_full,
                                         geo_norm=geo107_win_full)
                rm107 = _rmse(th107w.cpu().numpy().flatten(),
                              th_pred107.cpu().numpy().flatten())

            t_108_win = data_loader.df108["t_h"].values[m108_win]
            obs_108_np = th108w.cpu().numpy().flatten()
            persist_rmse_24h = _persistence_24h_rmse(obs_108_np, t_108_win)
            skill = float(np.clip(1.0 - rm108 / max(persist_rmse_24h, 1e-6), -2.0, 2.0))

        res = dict(
            t_start_h=float(t_start), t_end_h=float(t_end),
            fos_107=fos_107_val, fos_108=fos_108_val,
            fos_min=fos_min_sensed, fos_min_interp=fos_min_interp,
            rmse=rm108, rmse_108=rm108, rmse_107=rm107,
            persist_rmse_24h=persist_rmse_24h, skill=skill,
            warning=fos_min_sensed < fos_warn_thresh,
            failure=fos_min_sensed < 1.0,
            n_pts_107=n107, n_pts_108=len(x108w),
            geo107_mid=g107_mid, geo108_mid=g108_mid,
        )
        window_results.append(res)

        status = "🔴 FAIL" if res["failure"] else ("🟡 WARN" if res["warning"] else "🟢 OK")
        print(f"  Win{wi+1:02d} [{t_start:.0f}–{t_end:.0f}h]"
              f"  FoS107={fos_107_val:.3f}  FoS108={fos_108_val:.3f}"
              f"  FoS_min={fos_min_sensed:.3f}  {status}"
              f"  RMSE108={rm108:.4f}  Skill={skill:+.3f}"
              f"  geo107={g107_mid:.3f}  geo108={g108_mid:.3f}")

    if window_results:
        worst  = min(window_results, key=lambda r: r["fos_min"])
        n_warn = sum(r["warning"] for r in window_results)
        n_fail = sum(r["failure"] for r in window_results)
        skills = np.array([r["skill"] for r in window_results])
        print(f"\n[Stage2] windows={len(window_results)}  warnings={n_warn}  failures={n_fail}")
        print(f"  Worst FoS={worst['fos_min']:.3f} at t=[{worst['t_start_h']:.0f},{worst['t_end_h']:.0f}]h")
        print(f"  Skill: mean={skills.mean():+.3f}  min={skills.min():+.3f}  max={skills.max():+.3f}")
    return window_results


# ═══════════════════════════════════════════════════════════════════════════
#  FIGURE HELPERS  (geo-aware _pred_ts and _pred_fos_ts)
# ═══════════════════════════════════════════════════════════════════════════

matplotlib.rcParams.update({
    "font.family":"DejaVu Sans","font.size":11,
    "axes.titlesize":12,"axes.labelsize":11,
    "xtick.labelsize":10,"ytick.labelsize":10,
    "legend.fontsize":9,"figure.dpi":150,
})

_OUT = "iot_figures"
os.makedirs(_OUT, exist_ok=True)

def _save(fig, name):
    p = os.path.join(_OUT, name)
    fig.savefig(p, dpi=300, bbox_inches="tight", facecolor="white")
    print(f"  [Fig] {name}"); plt.close(fig)

def _pred_ts(model, df, x_norm_val, z_norm_val, batch=4096,
             rain_t_h=None, rain_q_ms=None, q_rain_max=None,
             geo_t_h=None, geo_norm_arr=None):
    t_np = df["t_h"].values.astype(np.float32)
    pred = np.zeros(len(t_np), dtype=np.float32)
    with torch.no_grad():
        for i in range(0, len(t_np), batch):
            t_b = torch.tensor(t_np[i:i+batch]/T_MAX_TRAIN,
                               dtype=torch.float32, device=device).unsqueeze(1)
            x_b = torch.full_like(t_b, x_norm_val)
            z_b = torch.full_like(t_b, z_norm_val)
            if rain_t_h is not None and rain_q_ms is not None:
                q_np = np.interp(t_np[i:i+batch], rain_t_h, rain_q_ms,
                                 left=0.0, right=float(rain_q_ms[-1]))
                qmax = q_rain_max if q_rain_max else max(float(rain_q_ms.max()), 1e-9)
                q_b  = torch.tensor(q_np / qmax, dtype=torch.float32,
                                    device=device).unsqueeze(1)
            else:
                q_b = None
            if geo_t_h is not None and geo_norm_arr is not None:
                g_np = np.interp(t_np[i:i+batch], geo_t_h, geo_norm_arr,
                                 left=float(geo_norm_arr[0]),
                                 right=float(geo_norm_arr[-1]))
                geo_b = torch.tensor(g_np, dtype=torch.float32,
                                     device=device).unsqueeze(1)
            else:
                geo_b = None
            _, th, _ = model(x_b, z_b, t_b, q_b, geo_norm=geo_b)
            pred[i:i+batch] = th.cpu().numpy().flatten()
    return t_np, pred

def _pred_fos_ts(model, x_norm_val, t_h_arr,
                 geo_t_h=None, geo_norm_arr=None):
    t_np = t_h_arr.astype(np.float32)
    fos  = np.zeros(len(t_np), dtype=np.float32)
    with torch.no_grad():
        for i in range(0, len(t_np), 2048):
            t_b = torch.tensor(t_np[i:i+2048]/T_MAX_TRAIN,
                               dtype=torch.float32, device=device).unsqueeze(1)
            x_b = torch.full_like(t_b, x_norm_val)
            z_b = torch.ones_like(t_b)
            if geo_t_h is not None and geo_norm_arr is not None:
                g_np = np.interp(t_np[i:i+2048], geo_t_h, geo_norm_arr,
                                 left=float(geo_norm_arr[0]),
                                 right=float(geo_norm_arr[-1]))
                geo_b = torch.tensor(g_np, dtype=torch.float32,
                                     device=device).unsqueeze(1)
            else:
                geo_b = torch.zeros_like(t_b)
            _, _, fp = model(x_b, z_b, t_b, geo_norm=geo_b)
            fos[i:i+2048] = fp.cpu().numpy().flatten()
    return np.clip(fos, 0.1, 15.0)

def _persistence_pred(obs, t_h, window_h=24):
    pred = np.zeros_like(obs)
    dt   = np.median(np.diff(t_h)) if len(t_h) > 1 else 1.0
    lag  = max(1, int(window_h / dt))
    pred[lag:] = obs[:-lag]; pred[:lag] = obs[0]
    return pred

def _lstm_pred(obs_all, seq_len=24):
    try:
        n_tr = int(len(obs_all)*0.70)
        obs_tr = obs_all[:n_tr]
        X, Y = [], []
        for i in range(seq_len, n_tr):
            X.append(obs_tr[max(0,i-seq_len):i]); Y.append(obs_tr[i])
        if len(X) < 10: return np.full_like(obs_all, obs_all.mean())
        X = torch.tensor(np.array(X), dtype=torch.float32).unsqueeze(-1)
        Y = torch.tensor(np.array(Y), dtype=torch.float32).unsqueeze(-1)
        lstm = nn.LSTM(1, 32, batch_first=True); head = nn.Linear(32, 1)
        opt  = torch.optim.Adam(list(lstm.parameters())+list(head.parameters()), lr=1e-3)
        for _ in range(200):
            opt.zero_grad()
            out, _ = lstm(X); ((head(out[:,-1,:]) - Y)**2).mean().backward(); opt.step()
        pred = np.zeros(len(obs_all)); pred[:seq_len] = obs_all[:seq_len]
        lstm.eval()
        with torch.no_grad():
            for i in range(seq_len, len(obs_all)):
                xin = torch.tensor(pred[i-seq_len:i], dtype=torch.float32).unsqueeze(0).unsqueeze(-1)
                out, _ = lstm(xin); pred[i] = head(out[0,-1,:]).item()
        return pred
    except Exception: return np.full_like(obs_all, obs_all.mean())


# ═══════════════════════════════════════════════════════════════════════════
#  FIGURES 1–8  (geo passed to all _pred_ts / _pred_fos_ts calls)
# ═══════════════════════════════════════════════════════════════════════════

def _fig1_convergence(train_hist, all_seed_results):
    n_seeds = len(all_seed_results)
    fig, axes = plt.subplots(2, 3, figsize=(17, 9))
    fig.suptitle("Stage 1 training convergence (geo 4th input)", fontweight="bold", fontsize=13)
    colors = ["#2196F3","#FF5722","#4CAF50"]
    for si, res in enumerate(all_seed_results):
        h = res["hist"]; col = colors[si % len(colors)]; lbl = f"Seed {res['seed']}"
        def _sc(r):
            r2s = [r.get(f"r2_{s}_{d}",float("nan")) for s in ["tr","val"] for d in [108,107]]
            rms = [r.get(f"rmse_{s}_{d}",float("nan")) for s in ["tr","val"] for d in [108,107]]
            if not all(np.isfinite(v) for v in r2s+rms): return -np.inf
            return float(np.mean(r2s)) - 10.0*float(np.mean(rms))
        star = " ★" if _sc(res) == max(_sc(r) for r in all_seed_results) else ""
        if h.get("total"):   axes[0,0].semilogy(h["total"],  color=col, alpha=0.75, lw=1.2, label=lbl+star)
        if h.get("data"):    axes[0,1].semilogy(h["data"],   color=col, alpha=0.75, lw=1.2, label=lbl+star)
        if h.get("pde"):     axes[1,0].semilogy([max(v,1e-10) for v in h["pde"]], color=col, alpha=0.75, lw=1.2, label=lbl+star)
        if h.get("r2_val"):
            ep_v, r2_v = zip(*h["r2_val"])
            axes[1,1].plot(ep_v, r2_v, color=col, alpha=0.85, lw=1.4, marker="o", ms=3, label=lbl+star)
        if h.get("beta"):    axes[0,2].semilogy([max(v,1e-10) for v in h["beta"]], color=col, alpha=0.75, lw=1.2, label=lbl+star)
    for ax, ttl in zip([axes[0,0],axes[0,1],axes[0,2],axes[1,0],axes[1,1]],
                       ["Total loss","Data loss","Beta prior loss","PDE (Richards) loss","Val R² history"]):
        ax.set_title(ttl, fontsize=11); ax.set_xlabel("Epoch"); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
        if ttl != "Val R² history": ax.set_ylabel("Loss")
        else:
            ax.axhline(0.55, color="red", ls="--", lw=1.2, label="R²=0.55 gate")
            ax.set_ylabel("R²"); ax.set_ylim(-0.5, 1.05)
    axes[1,2].axis("off")
    plt.tight_layout(); _save(fig, "fig1_convergence.png")

def _fig2_theta_timeseries(model, data_loader):
    df108 = data_loader.df108; df107 = data_loader.df107
    t_rain_h    = data_loader.t_rain_h
    q_rain_mmhr = data_loader.q_rain_ms * 3.6e6

    t108_h, pred108 = _pred_ts(model, df108, X_NORM_108, data_loader.z_norm_108,
                               rain_t_h=data_loader.t_rain_h,
                               rain_q_ms=data_loader.q_rain_ms,
                               q_rain_max=data_loader.q_rain_max,
                               geo_t_h=data_loader.t_geo108,
                               geo_norm_arr=data_loader.geo_norm108)
    t107_h, pred107 = _pred_ts(model, df107, X_NORM_107, data_loader.z_norm_107,
                               rain_t_h=data_loader.t_rain_h,
                               rain_q_ms=data_loader.q_rain_ms,
                               q_rain_max=data_loader.q_rain_max,
                               geo_t_h=data_loader.t_geo107,
                               geo_norm_arr=data_loader.geo_norm107)
    obs108 = df108["theta"].values; obs107 = df107["theta"].values
    tr108, val108, te108 = _splits(t108_h)
    tr107, val107, te107 = _splits(t107_h)

    fig = plt.figure(figsize=(16, 13))
    gs_ = _gs.GridSpec(5, 1, height_ratios=[0.6, 0.6, 2.5, 2.5, 0.8], hspace=0.42)
    fig.suptitle("Volumetric water content θ: observed vs PINN predicted (geo 4th input)",
                 fontweight="bold", fontsize=12)

    ax0 = fig.add_subplot(gs_[0])
    ax0.bar(t_rain_h, q_rain_mmhr, width=2, color="#4A90D9", alpha=0.8)
    ax0.set_ylabel("Rain (mm/hr)"); ax0.set_xlim(0, T_MAX)
    ax0.axvline(T_MAX_SYNC, color="purple", ls=":", lw=1.5)
    ax0.axvspan(T_MAX_SYNC*VAL_LO, T_MAX_SYNC*VAL_HI, alpha=0.12, color="#FF9800")
    ax0.set_title("Rainfall", fontsize=10)

    # Geo panel
    ax_geo = fig.add_subplot(gs_[1])
    ax_geo.plot(data_loader.t_geo108, data_loader.geo_norm108,
                color="#7B3F9E", lw=0.7, alpha=0.8, label="Dev108 geo_norm")
    ax_geo.plot(data_loader.t_geo107, data_loader.geo_norm107,
                color="#4A90D9", lw=0.7, alpha=0.7, label="Dev107 geo_norm")
    ax_geo.set_ylabel("geo_norm"); ax_geo.set_xlim(0, T_MAX)
    ax_geo.axvline(T_MAX_SYNC, color="purple", ls=":", lw=1.5)
    ax_geo.axvspan(T_MAX_SYNC*VAL_LO, T_MAX_SYNC*VAL_HI, alpha=0.12, color="#FF9800")
    ax_geo.legend(fontsize=7); ax_geo.set_title("Normalised geo input (log1p)", fontsize=10)

    for ax, obs, pred, t_h, tr_m, val_m, te_m, did, soil, depth in [
        (fig.add_subplot(gs_[2]), obs108, pred108, t108_h, tr108, val108, te108,
         108, "Sandy Clay Loam", "30 cm"),
        (fig.add_subplot(gs_[3]), obs107, pred107, t107_h, tr107, val107, te107,
         107, "Sandy Clay", "22 cm"),
    ]:
        if val_m.any():
            ax.axvspan(t_h[val_m][0], t_h[val_m][-1], alpha=0.10, color="#FF9800")
        if te_m.any():
            ax.axvspan(t_h[te_m][0], t_h[te_m][-1], alpha=0.06, color="#E91E63")
        ax.plot(t_h, obs, color="#333", lw=0.9, alpha=0.9, label="Observed θ")
        ax.plot(t_h, pred, color="#E91E63", lw=1.3, alpha=0.85, ls="--", label="PINN predicted θ")
        for mask, sname, col in [(tr_m,"Train","#2196F3"),(val_m,"Val","#FF9800"),(te_m,"Test","#E91E63")]:
            if mask.sum() > 2:
                r2 = _r2(obs[mask], pred[mask]); rm = _rmse(obs[mask], pred[mask])
                mid = t_h[mask].mean(); ypos = obs.max() + 0.01
                ax.text(mid, ypos, f"{sname}\nR²={r2:.3f}\nRMSE={rm:.4f}",
                        ha="center", va="bottom", fontsize=7.5, color=col,
                        bbox=dict(fc="white", ec=col, alpha=0.7, pad=2))
        ax.axvline(T_MAX_SYNC, color="purple", ls=":", lw=1.5)
        ax.set_ylabel("θ (m³/m³)"); ax.set_xlim(0, T_MAX)
        ax.set_title(f"Dev{did} — {soil}, x={SITE[did]['x_pos']:.0f} m, z={depth}", fontsize=10)
        ax.legend(fontsize=8)

    ax3 = fig.add_subplot(gs_[4])
    res108 = pred108 - obs108; res107 = pred107[:len(obs107)] - obs107
    ax3.plot(t108_h, res108, color="#D45F5F", lw=0.8, alpha=0.8, label="Residual Dev108")
    ax3.plot(t107_h, res107, color="#4A90D9", lw=0.8, alpha=0.7, label="Residual Dev107")
    ax3.axhline(0, color="k", lw=0.8, ls="--")
    ax3.axhspan(-0.03, 0.03, alpha=0.06, color="gray")
    ax3.axvspan(T_MAX_SYNC*VAL_LO, T_MAX_SYNC*VAL_HI, alpha=0.08, color="#FF9800")
    ax3.set_ylabel("Residual"); ax3.set_xlabel("Time (h)")
    ax3.set_xlim(0, T_MAX); ax3.set_ylim(-0.08, 0.08)
    ax3.legend(fontsize=8)

    _save(fig, "fig2_theta_timeseries.png")
    return (t108_h, pred108, obs108, tr108, val108, te108,
            t107_h, pred107, obs107, tr107, val107, te107)

def _fig3_scatter_residuals(t108_h, pred108, obs108, tr108, val108, te108,
                             t107_h, pred107, obs107, tr107, val107, te107):
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    fig.suptitle("Predicted vs observed scatter and residual distributions",
                 fontweight="bold", fontsize=13)
    for col_i, (sname, m108, m107, col) in enumerate([
        ("Train", tr108, tr107, "#2196F3"),
        ("Val",   val108,val107,"#FF9800"),
        ("Test",  te108, te107, "#E91E63"),
    ]):
        oc = np.concatenate([obs108[m108], obs107[m107]])
        pc = np.concatenate([pred108[m108], pred107[:len(obs107)][m107]])
        if len(oc) == 0: continue
        r2 = _r2(oc,pc); rm = _rmse(oc,pc)
        mn = min(oc.min(),pc.min())-0.005; mx = max(oc.max(),pc.max())+0.005
        ax = axes[0,col_i]
        ax.scatter(oc,pc,s=6,alpha=0.45,color=col)
        ax.plot([mn,mx],[mn,mx],"k--",lw=1.2)
        ax.set_xlabel("Observed θ"); ax.set_ylabel("Predicted θ")
        ax.set_title(f"{sname}  R²={r2:.4f}  RMSE={rm:.5f}\n(n={len(oc)})", fontsize=10)
        ax.set_xlim(mn,mx); ax.set_ylim(mn,mx); ax.set_aspect("equal"); ax.grid(True,alpha=0.3)
        ax2 = axes[1,col_i]
        res = pc-oc
        ax2.hist(res,bins=40,color=col,alpha=0.75,edgecolor="white")
        ax2.axvline(0,color="k",lw=1.2,ls="--")
        ax2.axvline(res.mean(),color="red",lw=1.2,label=f"mean={res.mean():.4f}")
        ax2.axvline(res.mean()+res.std(),color="gray",lw=0.9,ls=":",label=f"±1σ={res.std():.4f}")
        ax2.axvline(res.mean()-res.std(),color="gray",lw=0.9,ls=":")
        ax2.set_xlabel("Residual"); ax2.set_ylabel("Count")
        ax2.set_title(f"Residual distribution — {sname}", fontsize=10)
        ax2.legend(fontsize=8); ax2.grid(True,alpha=0.3)
    plt.tight_layout(); _save(fig, "fig3_scatter_residuals.png")

def _fig4_vg_curves(model):
    p = model.get_learned_params()
    psi_range = np.linspace(-8, 0, 500)
    def vg_theta_np(psi, alpha, n, theta_r, theta_s):
        m = 1.0-1.0/n; arg = np.abs(psi)*alpha
        Se = np.where(psi>=0, 1.0, 1.0/(1.0+arg**n)**m)
        Se = np.clip(Se,1e-6,1.0)
        return theta_r + (theta_s-theta_r)*Se
    fig, axes = plt.subplots(1, 2, figsize=(13, 6))
    fig.suptitle("Van Genuchten retention curves: PTF prior vs PINN-learned",
                 fontweight="bold", fontsize=13)
    for did, title, ax in [(107,"Dev107 Sandy Clay (x=25m)",axes[0]),
                            (108,"Dev108 Sandy Clay Loam (x=7m)",axes[1])]:
        s = SITE[did]
        th_ptf   = vg_theta_np(psi_range, s["alpha"],       s["n_vg"],      s["theta_r"],      s["theta_s"])
        th_learn = vg_theta_np(psi_range, p[f"alpha_{did}"],p[f"n_vg_{did}"],p[f"theta_r_{did}"],p[f"theta_s_{did}"])
        bounds   = PTF_BOUNDS[did]
        th_lo    = vg_theta_np(psi_range,bounds["alpha"][0],bounds["n_vg"][0],bounds["theta_r"][0],bounds["theta_s"][0])
        th_hi    = vg_theta_np(psi_range,bounds["alpha"][1],bounds["n_vg"][1],bounds["theta_r"][1],bounds["theta_s"][1])
        ax.fill_between(psi_range,th_lo,th_hi,alpha=0.15,color="#2196F3",label="PTF ±20–30% bounds")
        ax.plot(psi_range,th_ptf,"b--",lw=1.8,label="PTF prior")
        ax.plot(psi_range,th_learn,"r-",lw=2.2,label="PINN-learned")
        b_val = p[f"beta_{did}"]
        ax.set_xlabel("Matric potential ψ (m)"); ax.set_ylabel("θ (m³/m³)")
        ax.set_title(title, fontsize=10); ax.legend(fontsize=8); ax.grid(True,alpha=0.3)
        ax.set_xlim(-8, 0.2)
        txt = (f"Learned: α={p[f'alpha_{did}']:.3f}  n={p[f'n_vg_{did}']:.4f}\n"
               f"         θ_r={p[f'theta_r_{did}']:.4f}  θ_s={p[f'theta_s_{did}']:.4f}\n"
               f"Prior:   α={s['alpha']:.3f}  n={s['n_vg']:.4f}\n"
               f"         θ_r={s['theta_r']:.4f}  θ_s={s['theta_s']:.4f}\n"
               f"β(geo-Ks): {b_val:.4f}  "
               f"Ks_eff(g=1)={s['Ks']*(1+b_val):.3e} m/s")
        ax.text(0.02,0.98,txt,transform=ax.transAxes,fontsize=7.5,va="top",
                family="monospace",bbox=dict(fc="white",ec="gray",alpha=0.8,pad=3))
    plt.tight_layout(); _save(fig, "fig4_vg_curves.png")

def _fig5_fos_warning(model, data_loader, window_results):
    t_rain_h    = data_loader.t_rain_h
    q_rain_mmhr = data_loader.q_rain_ms * 3.6e6
    fos108 = _pred_fos_ts(model, X_NORM_108, data_loader.df108["t_h"].values,
                          geo_t_h=data_loader.t_geo108, geo_norm_arr=data_loader.geo_norm108)
    fos107 = _pred_fos_ts(model, X_NORM_107, data_loader.df107["t_h"].values,
                          geo_t_h=data_loader.t_geo107, geo_norm_arr=data_loader.geo_norm107)
    wins = window_results
    t_mid       = np.array([0.5*(w["t_start_h"]+w["t_end_h"]) for w in wins]) if wins else np.array([])
    fos_sensed  = np.array([w["fos_min"]        for w in wins]) if wins else np.array([])
    fos_interp  = np.array([w["fos_min_interp"] for w in wins]) if wins else np.array([])
    skill       = np.array([w["skill"]           for w in wins]) if wins else np.array([])

    fig = plt.figure(figsize=(15,14))
    gs_ = _gs.GridSpec(4,1,height_ratios=[1,2.5,2.5,1.5],hspace=0.42)
    fig.suptitle("Real-Time FoS Early Warning (geo 4th input)", fontweight="bold", fontsize=12)

    ax0 = fig.add_subplot(gs_[0])
    ax0.bar(t_rain_h, q_rain_mmhr, width=1.5, color="#4A90D9", alpha=0.8)
    ax0.set_ylabel("Rainfall (mm/hr)"); ax0.set_xlim(0, T_MAX)
    ax0.axvline(T_MAX_SYNC, color="purple", ls=":", lw=1.5)

    ax1 = fig.add_subplot(gs_[1])
    ax1.plot(data_loader.df108["t_h"].values, fos108, color="#D45F5F", lw=1.4, label="FoS Dev108 ✓")
    ax1.plot(data_loader.df107["t_h"].values, fos107, color="#4A90D9", lw=1.4, label="FoS Dev107 ✓", alpha=0.85)
    if len(t_mid):
        ax1.plot(t_mid, fos_sensed, color="#1a1a1a", lw=2.2, ls="-.", marker="D", ms=5,
                 label="Min FoS (sensor) ✓", zorder=5)
        ax1.plot(t_mid, fos_interp, color="#888", lw=1.2, ls=":", marker="x", ms=4,
                 label="Min FoS (interp) — unvalidated", alpha=0.7)
    ax1.axhline(1.5, color="orange", ls="--", lw=2.0, label="Warn 1.5")
    ax1.axhline(1.0, color="red",    ls="--", lw=2.5, label="Fail 1.0")
    ax1.axvline(T_MAX_SYNC, color="purple", ls=":", lw=1.5)
    ax1.set_ylabel("FoS"); ax1.set_xlim(0, T_MAX); ax1.legend(fontsize=8, ncol=2)

    ax2 = fig.add_subplot(gs_[2])
    if wins:
        ax2.plot(t_mid, fos_sensed, "rs-", ms=6, lw=1.5, label="FoS_min sensor ✓")
        ax2.plot(t_mid, fos_interp, "x--", color="gray", ms=5, lw=1.0, label="FoS_min interp", alpha=0.7)
        for w in wins:
            c = "red" if w["failure"] else ("orange" if w["warning"] else None)
            if c: ax2.axvspan(w["t_start_h"], w["t_end_h"], alpha=0.15, color=c)
        ax2.axhline(1.5, color="orange", ls="--", lw=2.0)
        ax2.axhline(1.0, color="red",    ls="--", lw=2.5)
        ax2.set_ylabel("FoS"); ax2.legend(fontsize=8); ax2.set_xlim(0, T_MAX)

    ax3 = fig.add_subplot(gs_[3])
    if wins and len(skill):
        bar_col = ["#2ecc71" if s > 0 else "#e74c3c" for s in skill]
        bw = 0.4*(t_mid[1]-t_mid[0]) if len(t_mid) > 1 else 100
        ax3.bar(t_mid, skill, width=bw, color=bar_col, alpha=0.85)
        ax3.axhline(0, color="k", lw=1.0)
        ax3.set_ylabel("Skill vs 24h persist"); ax3.set_xlabel("Time (h)")
        ax3.set_xlim(0, T_MAX); ax3.set_ylim(-2.1, 2.1)
    _save(fig, "fig5_fos_warning_fixed.png")

def _fig6_event_zoom(model, data_loader):
    t_rain_h    = data_loader.t_rain_h
    q_rain_mmhr = data_loader.q_rain_ms * 3.6e6
    rain_smooth = np.convolve(q_rain_mmhr, np.ones(24)/24, mode="same")
    peak_idxs = []
    used = set()
    for idx in np.argsort(rain_smooth)[::-1]:
        if all(abs(idx-j) > 200 for j in used):
            peak_idxs.append(idx); used.add(idx)
        if len(peak_idxs) == 3: break
    fig, axes = plt.subplots(3, 3, figsize=(16, 12))
    fig.suptitle("Event-level zoom: rainfall → θ → FoS (geo 4th input)", fontweight="bold", fontsize=13)
    for row, pidx in enumerate(peak_idxs):
        t_peak = t_rain_h[pidx]
        t_lo = max(0, t_peak - 120); t_hi = min(T_MAX_FULL, t_peak + 180)
        ax0 = axes[row,0]
        m_r = (t_rain_h >= t_lo) & (t_rain_h <= t_hi)
        ax0.bar(t_rain_h[m_r], q_rain_mmhr[m_r], width=1.5, color="#4A90D9", alpha=0.85)
        ax0.axvline(t_peak, color="red", lw=1.2, ls="--")
        ax0.set_title(f"Event {row+1}: rain (t={t_peak:.0f}h)", fontsize=9)
        ax0.set_xlabel("Time (h)"); ax0.set_ylabel("mm/hr")
        ax1 = axes[row,1]
        for df_, xn, zn, geo_t, geo_n, col, lbl in [
            (data_loader.df108, X_NORM_108, data_loader.z_norm_108,
             data_loader.t_geo108, data_loader.geo_norm108, "#D45F5F","Dev108"),
            (data_loader.df107, X_NORM_107, data_loader.z_norm_107,
             data_loader.t_geo107, data_loader.geo_norm107, "#4A90D9","Dev107"),
        ]:
            m_d = (df_["t_h"].values >= t_lo) & (df_["t_h"].values <= t_hi)
            if m_d.sum() < 2: continue
            df_win = df_[m_d]
            t_win, pred_win = _pred_ts(model, df_win, xn, zn,
                                        rain_t_h=data_loader.t_rain_h,
                                        rain_q_ms=data_loader.q_rain_ms,
                                        q_rain_max=data_loader.q_rain_max,
                                        geo_t_h=geo_t, geo_norm_arr=geo_n)
            ax1.plot(df_win["t_h"].values, df_win["theta"].values,
                     color=col, lw=1.0, alpha=0.8, label=f"{lbl} obs")
            ax1.plot(t_win, pred_win, color=col, lw=1.5, ls="--", label=f"{lbl} pred")
        ax1.axvline(t_peak, color="red", lw=1.2, ls="--")
        ax1.set_title(f"Event {row+1}: θ", fontsize=9)
        ax1.set_xlabel("Time (h)"); ax1.set_ylabel("θ"); ax1.legend(fontsize=7); ax1.grid(True,alpha=0.3)
        ax2 = axes[row,2]
        t_fine = np.linspace(t_lo, t_hi, 400)
        fos108w = _pred_fos_ts(model, X_NORM_108, t_fine,
                               geo_t_h=data_loader.t_geo108, geo_norm_arr=data_loader.geo_norm108)
        fos107w = _pred_fos_ts(model, X_NORM_107, t_fine,
                               geo_t_h=data_loader.t_geo107, geo_norm_arr=data_loader.geo_norm107)
        ax2.plot(t_fine, fos108w, color="#D45F5F", lw=1.5, label="FoS Dev108")
        ax2.plot(t_fine, fos107w, color="#4A90D9", lw=1.5, label="FoS Dev107")
        ax2.axhline(1.5, color="orange", ls="--", lw=1.5)
        ax2.axhline(1.0, color="red",    ls="--", lw=2.0)
        ax2.axvline(t_peak, color="red", lw=1.2, ls="--")
        ax2.set_title(f"Event {row+1}: FoS", fontsize=9)
        ax2.set_xlabel("Time (h)"); ax2.set_ylabel("FoS")
        ax2.set_ylim(0.5, 6.0); ax2.legend(fontsize=7); ax2.grid(True,alpha=0.3)
    plt.tight_layout(); _save(fig, "fig6_event_zoom.png")

def _fig7_spatial_fos(model, data_loader):
    t_rain_h    = data_loader.t_rain_h
    q_rain_mmhr = data_loader.q_rain_ms * 3.6e6
    rain_smooth = np.convolve(q_rain_mmhr, np.ones(12)/12, mode="same")
    dry_times   = t_rain_h[np.argsort(rain_smooth)[:3]]
    wet_times   = t_rain_h[np.argsort(rain_smooth)[-3:]]
    t_stamps    = np.sort(np.concatenate([dry_times[:2], wet_times[-2:],
                          [T_MAX_SYNC*0.4, T_MAX_SYNC*0.7]]))
    t_stamps    = np.clip(t_stamps, 10, T_MAX_SYNC-10)
    x_scan  = np.linspace(X_NORM_108, X_NORM_107, 40)
    x_m     = x_scan * X_MAX
    fig, axes = plt.subplots(2, 3, figsize=(15, 9))
    fig.suptitle("Spatial FoS profile along slope (geo 4th input)", fontweight="bold", fontsize=13)
    cmap = plt.cm.coolwarm_r
    for i, (ax, t_h) in enumerate(zip(axes.flat, t_stamps)):
        x_t = torch.tensor(x_scan, dtype=torch.float32, device=device).unsqueeze(1)
        z_t = torch.ones_like(x_t)
        t_t = torch.full_like(x_t, t_h / T_MAX_TRAIN)
        # Spatially blend geo at this timestamp
        g107_t = float(np.interp(t_h, data_loader.t_geo107, data_loader.geo_norm107))
        g108_t = float(np.interp(t_h, data_loader.t_geo108, data_loader.geo_norm108))
        alp    = np.clip((x_scan - X_NORM_108)/(X_NORM_107 - X_NORM_108 + 1e-9), 0., 1.)
        geo_blend = torch.tensor((1-alp)*g108_t + alp*g107_t,
                                 dtype=torch.float32, device=device).unsqueeze(1)
        with torch.no_grad():
            _, _, fos_scan = model(x_t, z_t, t_t, geo_norm=geo_blend)
        fos_np = fos_scan.cpu().numpy().flatten()
        rain_at_t = float(np.interp(t_h, t_rain_h, q_rain_mmhr))
        col = cmap(np.clip((5.0 - fos_np.mean()) / 4.0, 0, 1))
        ax.plot(x_m, fos_np, color=col, lw=2.0)
        ax.fill_between(x_m, fos_np, 1.0, where=fos_np < 1.5, alpha=0.2, color="orange")
        ax.fill_between(x_m, fos_np, 1.0, where=fos_np < 1.0, alpha=0.3, color="red")
        ax.axhline(1.5, color="orange", ls="--", lw=1.2)
        ax.axhline(1.0, color="red",    ls="--", lw=1.8)
        ax.axvline(SITE[108]["x_pos"], color="#D45F5F", lw=1.0, ls=":")
        ax.axvline(SITE[107]["x_pos"], color="#4A90D9", lw=1.0, ls=":")
        ax.scatter([SITE[108]["x_pos"], SITE[107]["x_pos"]],
                   [float(np.interp(SITE[108]["x_pos"],x_m,fos_np)),
                    float(np.interp(SITE[107]["x_pos"],x_m,fos_np))],
                   s=60, zorder=5, color=["#D45F5F","#4A90D9"], marker="D")
        ax.set_xlim(x_m[0]-0.5, x_m[-1]+0.5)
        ax.set_ylim(0.5, min(fos_np.max()+0.5, 8.0))
        ax.set_xlabel("x (m)"); ax.set_ylabel("FoS")
        geo_str = f"g107={g107_t:.2f} g108={g108_t:.2f}"
        ax.set_title(f"t={t_h:.0f}h  rain={rain_at_t:.2f}mm/hr\n"
                     f"FoS∈[{fos_np.min():.2f},{fos_np.max():.2f}]  {geo_str}", fontsize=9)
        ax.grid(True, alpha=0.3)
    plt.tight_layout(); _save(fig, "fig7_spatial_fos.png")

def _fig8_baseline_comparison(model, data_loader):
    df108 = data_loader.df108
    t108_h, pred_pinn = _pred_ts(model, df108, X_NORM_108, data_loader.z_norm_108,
                                  rain_t_h=data_loader.t_rain_h,
                                  rain_q_ms=data_loader.q_rain_ms,
                                  q_rain_max=data_loader.q_rain_max,
                                  geo_t_h=data_loader.t_geo108,
                                  geo_norm_arr=data_loader.geo_norm108)
    obs108 = df108["theta"].values
    tr108, val108, te108 = _splits(t108_h)
    pred_pers = _persistence_pred(obs108, t108_h, window_h=24)
    pred_lstm = _lstm_pred(obs108)

    fig, axes = plt.subplots(3, 2, figsize=(14, 12))
    fig.suptitle("Forecast comparison: PINN(geo) vs baselines (Dev108)",
                 fontweight="bold", fontsize=13)
    methods = [("PINN+geo (this work)", pred_pinn, "#E91E63"),
               ("24h persistence",      pred_pers, "#FF9800"),
               ("LSTM (data-only)",     pred_lstm, "#4CAF50")]
    split_defs = [("Train",tr108),("Val",val108),("Test",te108)]

    for mi, (mname, pred, col) in enumerate(methods):
        ax = axes[0, min(mi, 1)]
        if mi == 2:
            ax2_twin = axes[0,1].twinx()
            ax2_twin.plot(t108_h, pred, color=col, lw=1.0, alpha=0.6, ls=":", label=mname)
            continue
        ax.plot(t108_h, obs108, color="#333", lw=0.8, alpha=0.8, label="Observed")
        ax.plot(t108_h, pred, color=col, lw=1.3, ls="--", label=mname)
        ax.axvspan(T_MAX_SYNC*VAL_LO, T_MAX_SYNC*VAL_HI, alpha=0.10, color="#FF9800")
        ax.set_title(mname, fontsize=10); ax.set_xlabel("Time (h)"); ax.set_ylabel("θ")
        ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

    ax_bar = axes[1,0]; x_pos = np.arange(3); bar_w = 0.22
    for mi, (mname, pred, col) in enumerate(methods):
        r2s = [_r2(obs108[m],pred[m]) if m.sum()>2 else float("nan") for _,m in split_defs]
        ax_bar.bar(x_pos + mi*bar_w, r2s, width=bar_w, color=col, alpha=0.8, label=mname)
    ax_bar.set_xticks(x_pos+bar_w); ax_bar.set_xticklabels(["Train","Val\n(mid 30%)","Test"])
    ax_bar.axhline(0.55, color="red", ls="--", lw=1.2)
    ax_bar.set_ylabel("R²"); ax_bar.set_title("R² by split", fontsize=10)
    ax_bar.legend(fontsize=7); ax_bar.grid(True, alpha=0.3, axis="y")

    ax_rm = axes[1,1]
    for mi, (mname, pred, col) in enumerate(methods):
        rms = [_rmse(obs108[m],pred[m]) if m.sum()>2 else float("nan") for _,m in split_defs]
        ax_rm.bar(x_pos + mi*bar_w, rms, width=bar_w, color=col, alpha=0.8, label=mname)
    ax_rm.set_xticks(x_pos+bar_w); ax_rm.set_xticklabels(["Train","Val\n(mid 30%)","Test"])
    ax_rm.axhline(0.03, color="red", ls="--", lw=1.2)
    ax_rm.set_ylabel("RMSE"); ax_rm.set_title("RMSE by split", fontsize=10)
    ax_rm.legend(fontsize=7); ax_rm.grid(True, alpha=0.3, axis="y")

    ax_sk = axes[2,0]
    skills_test = []
    for mname, pred, col in methods:
        if te108.sum() > 2:
            rm_m = _rmse(obs108[te108], pred[te108])
            rm_p = _persistence_24h_rmse(obs108[te108], t108_h[te108])
            sk   = float(np.clip(1.0 - rm_m / max(rm_p, 1e-6), -2, 2))
        else:
            sk = float("nan")
        skills_test.append((mname, sk, col))
    names_sk = [s[0] for s in skills_test]; vals_sk = [s[1] for s in skills_test]
    cols_sk  = [s[2] for s in skills_test]
    bars = ax_sk.bar(names_sk, vals_sk, color=cols_sk, alpha=0.8)
    ax_sk.axhline(0, color="k", lw=1.0)
    for bar, v in zip(bars, vals_sk):
        if np.isfinite(v):
            ax_sk.text(bar.get_x()+bar.get_width()/2, v+0.03, f"{v:+.3f}", ha="center", fontsize=9)
    ax_sk.set_ylabel("Skill vs 24h persistence (test)"); ax_sk.set_title("Skill score", fontsize=10)
    ax_sk.set_ylim(-1.5, 1.5); ax_sk.grid(True, alpha=0.3, axis="y")

    ax_sc = axes[2,1]
    for mname, pred, col in methods:
        if te108.sum() > 2:
            ax_sc.scatter(obs108[te108], pred[te108], s=8, alpha=0.5, color=col, label=mname)
    mn = obs108[te108].min()-0.005 if te108.any() else 0.08
    mx = obs108[te108].max()+0.005 if te108.any() else 0.39
    ax_sc.plot([mn,mx],[mn,mx],"k--",lw=1.2)
    ax_sc.set_xlabel("Observed θ"); ax_sc.set_ylabel("Predicted θ")
    ax_sc.set_title("Scatter (test split)", fontsize=10)
    ax_sc.legend(fontsize=7); ax_sc.set_aspect("equal"); ax_sc.grid(True,alpha=0.3)
    plt.tight_layout(); _save(fig, "fig8_baseline_comparison.png")


def generate_all_figures(model, data_loader, window_results, train_hist,
                         all_seed_results=None):
    model.eval()
    print(f"\n{'─'*55}\n  Generating figures → {_OUT}/\n{'─'*55}")
    if all_seed_results is None:
        all_seed_results = [{"seed":0,"hist":train_hist,"score":1.0}]
    print("[Fig 1] Convergence..."); _fig1_convergence(train_hist, all_seed_results)
    print("[Fig 2] θ time-series..."); ts_data = _fig2_theta_timeseries(model, data_loader)
    print("[Fig 3] Scatter + residuals..."); _fig3_scatter_residuals(*ts_data)
    print("[Fig 4] VG retention curves..."); _fig4_vg_curves(model)
    print("[Fig 5] FoS early warning..."); _fig5_fos_warning(model, data_loader, window_results)
    print("[Fig 6] Event zoom..."); _fig6_event_zoom(model, data_loader)
    print("[Fig 7] Spatial FoS..."); _fig7_spatial_fos(model, data_loader)
    print("[Fig 8] Baseline comparison..."); _fig8_baseline_comparison(model, data_loader)
    print(f"\n[Done] 8 figures saved to {_OUT}/")


# ═══════════════════════════════════════════════════════════════════════════
#  MAIN
# ═══════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    bar = "="*65
    print(f"\n{bar}")
    print("  2D PINN — Slope Hydrology  [GEO 4TH INPUT VERSION]")
    print(f"  Inputs: (x, z, t, q_rain, geo_norm)")
    print(f"  K_s^eff = Ks*(1 + beta*geo_norm)  [learnable beta per device]")
    print(f"{bar}\n")

    if not os.path.exists(CSV_FILE):
        raise FileNotFoundError(f"CSV not found: {CSV_FILE}")

    data = SlopeDataLoader(CSV_FILE).load()

    print(f"\n{bar}\n  STAGE 1 — Physics Discovery\n{bar}")
    best_model, best_hist, all_results = train_stage1(
        data,
        n_seeds=2, total_epochs=12000, lr=2e-4,
        lam_data=500.0, lam_richards=1.0,
        lam_bc_top=0.1, lam_bc_bot=0.1,
        lam_ic=0.0, lam_smooth=0.01,
        lam_prior=0.05, lam_psi_var=1.0,
        lam_beta=0.01,          # beta prior — increase to 0.1 if overfitting to geo
        es_patience=10000, anneal_epochs=4000,
    )

    pth_s1 = "pinn_slope_geo_stage1.pth"
    torch.save({
        "state_dict":   best_model.state_dict(),
        "architecture": {"hidden":N_HIDDEN,"width":N_WIDTH,"dropout":DROPOUT,
                         "n_inputs":5},
        "site_params":  SITE,
        "stage":        "stage1_geo4th",
        "learned_vg":   best_model.get_learned_params(),
        "T_MAX_TRAIN":  T_MAX_TRAIN,
        "GEO_LOG_MIN":  GEO_LOG_MIN,
        "GEO_LOG_MAX":  GEO_LOG_MAX,
        "fixes_applied": ["causal_bc","train_norm","skill_score","sensor_fos_only",
                          "nan_tracking","raw_rain","middle_holdout_split",
                          "geo_4th_input","vg_K_eff","beta_prior"],
    }, pth_s1)
    print(f"[Saved] {pth_s1}")

    import shutil as _shutil
    _shutil.copy(pth_s1, os.path.join(DRIVE_CKPT_DIR, pth_s1))
    print(f"[Drive] {pth_s1} → {DRIVE_CKPT_DIR}/")

    print(f"\n{bar}\n  STAGE 2 — Rolling Window Adaptation\n{bar}")
    rolling_results = train_stage2(
        best_model, data,
        window_h=421.0, warmup_epochs=500, finetune_epochs=3000,
        lr=2e-4, lam_data=1000.0, lam_richards=0.5,
        lam_bc_top=0.1, lam_bc_bot=0.1,
        lam_smooth=0.01, lam_prior=0.5, lam_beta=0.01,
        fos_warn_thresh=1.5, es_patience=500,
    )

    pth_s2 = "pinn_slope_geo_stage2.pth"
    torch.save({
        "state_dict":      best_model.state_dict(),
        "site_params":     SITE,
        "stage":           "stage2_geo4th",
        "rolling_results": rolling_results,
        "learned_vg":      best_model.get_learned_params(),
        "T_MAX_TRAIN":     T_MAX_TRAIN,
        "GEO_LOG_MIN":     GEO_LOG_MIN,
        "GEO_LOG_MAX":     GEO_LOG_MAX,
    }, pth_s2)
    print(f"[Saved] {pth_s2}")
    _shutil.copy(pth_s2, os.path.join(DRIVE_CKPT_DIR, pth_s2))

    print(f"\n{bar}\n  GENERATING FIGURES\n{bar}")
    generate_all_figures(best_model, data, rolling_results, best_hist,
                         all_seed_results=all_results)

    print(f"\n{bar}\n  FINAL SUMMARY\n{bar}")
    m = compute_metrics_combined(best_model, data)
    print(f"  Test R²   = {m.get('r2_te',  float('nan')):.4f}")
    print(f"  Test RMSE = {m.get('rmse_te',float('nan')):.5f} m³/m³")

    p = best_model.get_learned_params()
    print(f"\n  Learned VG parameters:")
    print(f"  Dev107: α={p['alpha_107']:.3f}  n={p['n_vg_107']:.4f}"
          f"  θ_r={p['theta_r_107']:.4f}  θ_s={p['theta_s_107']:.4f}")
    print(f"  Dev108: α={p['alpha_108']:.3f}  n={p['n_vg_108']:.4f}"
          f"  θ_r={p['theta_r_108']:.4f}  θ_s={p['theta_s_108']:.4f}")

    print(f"\n  Learned vibration coupling (beta):")
    b107 = p['beta_107']; b108 = p['beta_108']
    print(f"  Dev107: β={b107:.4f}"
          f"  Ks_eff(geo=1) = {SITE[107]['Ks']*(1+b107):.3e} m/s"
          f"  (base={SITE[107]['Ks']:.3e})")
    print(f"  Dev108: β={b108:.4f}"
          f"  Ks_eff(geo=1) = {SITE[108]['Ks']*(1+b108):.3e} m/s"
          f"  (base={SITE[108]['Ks']:.3e})")
    if b107 < 0.05 and b108 < 0.05:
        print("  [Note] β≈0 — geo did not improve fit beyond rainfall at this site")
    elif b107 > 1.0 or b108 > 1.0:
        print("  [Note] Large β — strong vibration-Ks coupling detected")

    if rolling_results:
        skills = np.array([r["skill"] for r in rolling_results])
        worst  = min(rolling_results, key=lambda r: r["fos_min"])
        n_warn = sum(r["warning"] for r in rolling_results)
        n_fail = sum(r["failure"] for r in rolling_results)
        print(f"\n  FoS_min(sensor) = {worst['fos_min']:.3f}"
              f"  at t=[{worst['t_start_h']:.0f},{worst['t_end_h']:.0f}]h")
        print(f"  Warnings: {n_warn}   Failures: {n_fail}")
        print(f"  Skill (24h persist, Dev108): mean={skills.mean():+.3f}"
              f"  range=[{skills.min():.3f},{skills.max():.3f}]")
    print(bar)

[Drive] Checkpoint directory : /content/pinn_checkpoints
[Drive] Periodic interval    : every 2000 epochs
[Device] cpu

  2D PINN — Slope Hydrology  [GEO 4TH INPUT VERSION]
  Inputs: (x, z, t, q_rain, geo_norm)
  K_s^eff = Ks*(1 + beta*geo_norm)  [learnable beta per device]

[StratB] t=0 reset to t_orig=4.58h
[Fix2] T_MAX_TRAIN=297.1h
[Fix7] Train:[0,149h)∪(276h,424h]  Val:[149,276h]  Test:[361,424h]
[DataClean] Dev108: rescaled [0.4100,0.5400] -> [0.0630,0.3900]
[DataClean] Dev107: rescaled [0.2850,0.4630] -> [0.1000,0.3800]
[Geo Dev107] 9 dropout samples interpolated
[Geo Dev107] geo=[0.011,39.625] Hz
[Geo Dev108] 9 dropout samples interpolated
[Geo Dev108] geo=[0.035,35.222] Hz
[Geo] log1p norm: min=0.0105  max=3.7044  (training window 297h)
[Fix1] Causal rainfall BC set.
[Fix6] Rain function from raw record.
[Fix7] Rainfall in val window: 350.3 mm  ✓

[Data] Dev107: 1698 pts  θ=[0.100,0.378]  geo=[0.000,1.000]
[Data] Dev108: 1698 pts  θ=[0.063,0.386]  geo=[0.006,0.969]
[Data] T_MAX